# Bài 20 · GAN: hai mạng đấu nhau, một bên làm giả chữ số, một bên bắt giả

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/20-gan.ipynb)

Notebook đi kèm [Bài 20 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/gan/) (demo tương tác nằm ở trang bài học).
Ta tự viết bằng numpy một GAN: bộ sinh G biến nhiễu thành mẫu, bộ phân biệt D đoán mẫu thật hay giả, và hai mạng
học bằng cách đấu nhau. Câu hỏi trung tâm: **trò chơi này thật sự cực tiểu cái gì, và làm sao biết nó đã thắng?**

**Bạn sẽ làm:**
1. Viết hai mất mát của bộ sinh và thấy vì sao mất mát gốc $\log(1 - D(G(z)))$ bão hòa.
2. Tính chính xác bộ phân biệt tốt nhất $D^*$ trong một trò chơi một chiều, kiểm tra hai đẳng thức với JSD, và xem
   mỗi mất mát đưa bộ sinh tới đâu.
3. Huấn luyện GAN hai chiều trên một vòng 8 cụm, có và không có phạt gradient R1, và thấy mode collapse.
4. Sinh chữ số MNIST bằng bộ sinh của demo và đo khoảng cách từ mỗi mẫu tới ảnh huấn luyện gần nhất.
5. Đánh giá: vì sao $D \approx \tfrac12$ không có nghĩa là thành công (FD, precision/recall, một bộ phân loại mới).

**Cần biết trước:** [Bài 04 · Softmax & hàm mất mát](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/ham-mat-mat/)
(cross-entropy nhị phân), [Bài 06 · Lan truyền ngược](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/lan-truyen-nguoc/),
Adam của [Bài 07](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/bo-toi-uu/), và
[Bài 19 · Autoencoder & VAE](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/autoencoder-vae/) (bộ giải mã, thước đo
FD). **Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải MNIST (~11 MB) và 1,1 MB kết quả của demo lần đầu.

Mọi con số trong notebook đều do chính các ô bên dưới tính ra, và trùng với số trong demo. Riêng các lần huấn luyện
đầy đủ trên MNIST (15 cấu hình × 3 seed, mỗi lần 20.000 bước) được nạp từ đúng các tệp mà
demo dùng; notebook chạy lại một phiên bản nhỏ của từng cơ chế và kiểm tra những gì tính lại được.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi.

In [ ]:
import base64
import gzip
import hashlib
import itertools
import json
import math
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng điểm, nét, nhãn):
# dữ liệu thật = xanh, chấm tròn; mẫu của bộ sinh = cam, ô vuông; mất mát gốc = xanh lá, nét liền
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("cam_xam_xanh", [ORANGE, "#f2f2f2", BLUE])  # âm → 0 → dương
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


def span(values, digits=2):
    """Khoảng min–max qua các seed, như demo: 7,88–8,02."""
    lo, hi = min(values), max(values)
    return vn(lo, digits) if round(lo, digits) == round(hi, digits) else f"{vn(lo, digits)}–{vn(hi, digits)}"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sci(x, digits=1):
    """Số rất nhỏ hoặc rất lớn: 6,2·10⁻⁵."""
    e = math.floor(math.log10(abs(x)))
    return f"{vn(x / 10**e, digits)}·10{str(e).translate(SUP)}"


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 60.000 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    """Nhãn vạch trục log: 0,5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Đổi nhãn mọi trục của hình (cả thanh màu) sang kiểu Việt Nam; gọi ngay trước plt.show().
    Trục log lấy vạch 1–2–5 nếu ngắn, hoặc các lũy thừa của 10 (cách đều, tối đa ~9 vạch) nếu dài."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


def tile(images, rows, cols, gap=2):
    """Ghép rows × cols ảnh 28×28 thành một ảnh lớn (nhanh hơn nhiều so với một trục cho mỗi ảnh); nền khe = 0."""
    out = np.zeros((rows * (28 + gap) - gap, cols * (28 + gap) - gap))
    for i, im in enumerate(images[: rows * cols]):
        r, c = divmod(i, cols)
        out[r * (28 + gap) : r * (28 + gap) + 28, c * (28 + gap) : c * (28 + gap) + 28] = np.reshape(im, (28, 28))
    return out


def sigmoid(a):
    return 0.5 * (1 + np.tanh(0.5 * a))  # không tràn số với |a| lớn


def softplus(a):
    return np.maximum(a, 0) + np.log1p(np.exp(-np.abs(a)))  # log(1 + eᵃ), ổn định số


print(vn(13007, 0), vn(0.1, 1), vn(-4.3805, 4), pct(0.887), span([7.88, 8.02]), sci(6.2e-5))

## 2 · Dữ liệu

MNIST (LeCun và cộng sự, 1998): 60.000 ảnh huấn luyện và 10.000 ảnh kiểm tra, chữ số viết tay xám 28×28. Ô dưới tải
4 tệp gốc và kiểm tra mã MD5, nên ai chạy cũng có đúng cùng một dữ liệu. Như các lần chạy của demo (và Bài 19), GAN
học trên 55.000 ảnh huấn luyện đầu tiên; 10.000 ảnh kiểm tra chỉ dùng để đánh giá.

Ô cũng nạp bốn tệp kết quả mà demo dùng (`fetch_json`, có kiểm tra MD5):
- `gan-1d.json`: trò chơi một chiều với $D^*$ chính xác (mục 4);
- `gan-2d.json`: GAN hai chiều trên vòng 8 cụm, 5 seed × 20.000 bước cho mỗi cấu hình (mục 5);
- `gan-mnist.json`: mọi lần chạy GAN trên MNIST, 3 seed mỗi cấu hình, cùng các phép đo của chúng (mục 7–8);
- `gan-g16.json`: bộ sinh tốt nhất (seed 0), cùng định dạng với bộ giải mã của Bài 19 (mục 6).

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # mã MD5 chuẩn của MNIST
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def load(kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 784)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# Kết quả của các lần huấn luyện dài (cùng tệp với demo trên web). build.py thay COURSE_DATA_URL bằng thư mục data/
# của repo notebook công khai.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
one = fetch_json("gan-1d.json", "db1315153556b0b70774b9a9a4685439")
two = fetch_json("gan-2d.json", "b8b7c9bd3dee0ab24d66dcd40e5282a8")
mn = fetch_json("gan-mnist.json", "79bcc429034a8648347eb6999d9d27cd")
g16 = fetch_json("gan-g16.json", "fbb94a427bde0d1a5f3e042eeaf49c26")
size = sum((Path("data") / f).stat().st_size for f in ["gan-1d.json", "gan-2d.json", "gan-mnist.json", "gan-g16.json"])
print("huấn luyện:", train_img.shape, " kiểm tra:", test_img.shape, f" · 4 tệp của demo: {vn(size / 1e6, 1)} MB")

In [ ]:
Xtr = train_img[:55000].astype(np.float32) / 255  # 55.000 ảnh mà GAN học, điểm ảnh xám trong [0, 1]
Xte = test_img.astype(np.float32) / 255  # 10.000 ảnh kiểm tra
print(f"huấn luyện {vn(len(Xtr), 0)} · kiểm tra {vn(len(Xte), 0)} ảnh · độ sáng trung bình {pct(Xtr.mean())}")

## 3 · Hai mạng, hai mất mát

Một GAN có hai mạng. **Bộ sinh** $G$ biến một vectơ nhiễu $z \sim \mathcal N(0, I)$ thành một mẫu $G(z)$ (ở đây $z$ là
nhiễu đầu vào, như mã tiềm ẩn của Bài 19, không phải tổng có trọng số $z = xW + b$ của Bài 01). **Bộ phân biệt** $D$
nhận một mẫu $x$ và trả về một logit $a(x)$, với $D(x) = \sigma(a(x))$ là xác suất nó cho rằng $x$ là thật. $D$ học
như bộ phân loại nhị phân của Bài 04, nhãn 1 cho mẫu thật và 0 cho mẫu của $G$:

$$\mathcal L_D = \mathbb E_x\big[-\log D(x)\big] + \mathbb E_z\big[-\log(1 - D(G(z)))\big]
= \mathbb E_x\big[\mathrm{softplus}(-a(x))\big] + \mathbb E_z\big[\mathrm{softplus}(a(G(z)))\big]$$

với $\mathrm{softplus}(u) = \log(1 + e^u)$. Bộ sinh thì có hai lựa chọn:
- **mất mát gốc** của trò chơi minimax (Goodfellow và cộng sự, 2014): cực tiểu $\log(1 - D(G(z))) =
  -\mathrm{softplus}(a)$;
- **mất mát không bão hòa**, thứ mà gần như mọi người dùng: cực tiểu $-\log D(G(z)) = \mathrm{softplus}(-a)$.

Theo logit $a$ của một mẫu giả, gradient của hai mất mát là $-\sigma(a) = -D$ và $-(1 - \sigma(a)) = -(1 - D)$:
cùng hướng (tăng $a$, tức làm $D$ tin mẫu là thật), nhưng độ lớn khác nhau. Ô dưới kiểm tra hai đạo hàm đó bằng sai
phân hữu hạn, rồi vẽ độ lớn của chúng theo $D(G(z))$. Đường chấm đánh dấu giá trị $D(G(z))$ thật ở bước 500 của một
lần chạy MNIST với mất mát gốc (seed 0, đọc từ `gan-mnist.json`).

In [ ]:
D_fake = np.logspace(-6, 0, 401)[:-1]  # D(G(z)) từ 10⁻⁶ đến gần 1
a = np.log(D_fake) - np.log1p(-D_fake)  # logit, D = σ(a)
g_sat, g_ns = -sigmoid(a), -sigmoid(-a)  # ∂/∂a của −softplus(a) và của softplus(−a)
h = 1e-6
assert np.allclose((-softplus(a + h) + softplus(a - h)) / (2 * h), g_sat, rtol=1e-6, atol=1e-12)
assert np.allclose((softplus(-a - h) - softplus(-a + h)) / (2 * h), g_ns, rtol=1e-6, atol=1e-12)

sat_traj = mn["runs"]["sat-r0-lr0.001"]["seeds"][0]["traj"]
p500 = next(t for t in sat_traj if t["step"] == 500)
d500 = p500["DGz"]
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.loglog(D_fake, -g_ns, color=ORANGE, lw=2.5, ls="--", label="không bão hòa −log D: |∂/∂a| = 1 − D")
ax.loglog(D_fake, -g_sat, color=GREEN, lw=2.5, label="mất mát gốc log(1 − D): |∂/∂a| = D")
ax.axvline(d500, color=GREY, lw=1.2, ls=":")
ax.text(d500 * 1.4, 1e-2, "MNIST, mất mát gốc,\nbước 500", fontsize=8, color="black")
ax.set(xlabel="D(G(z)): xác suất D gán cho mẫu giả là thật", ylabel="độ lớn gradient theo logit a", ylim=(5e-7, 3))
ax.set_title("Khi D bác bỏ dễ dàng, mất mát gốc gần như phẳng")
ax.legend(frameon=False, fontsize=9, loc="lower right")
vn_axes(fig)
plt.show()
print(f"bước 500 (MNIST, mất mát gốc, seed 0): D(G(z)) trung bình = {sci(d500)}")
print(f"  một mẫu: |∂/∂a| gốc {sci(d500)}, không bão hòa {vn(1 - d500, 5)} → gấp {vn((1 - d500) / d500, 0)} lần")
print(f"  cả mạng G (chuẩn gradient trên 256 nhiễu cố định): không bão hòa {vn(p500['gNS'], 2)}, gốc {sci(p500['gSAT'])}"
      f" → gấp {vn(p500['gNS'] / p500['gSAT'], 0)} lần (≈ {vn(round(p500['gNS'] / p500['gSAT'], -3), 0)})")  # fmt: skip

Hai đường gặp nhau ở $D = \tfrac12$. Bên trái, khi $D$ chắc chắn mẫu là giả ($D(G(z)) \to 0$), gradient của mất mát
gốc tỉ lệ với chính $D$ và tắt dần, còn của mất mát không bão hòa thì gần 1. Đó đúng là lúc đầu huấn luyện: $G$ còn
tệ, $D$ dễ dàng nhận ra mẫu giả. Goodfellow và cộng sự viết rằng khi ấy $\log(1 - D(G(z)))$ *bão hòa*, và đề nghị
cho $G$ cực đại $\log D(G(z))$ thay vào: cùng một điểm cố định, nhưng gradient mạnh hơn nhiều lúc đầu.

Ở lần chạy MNIST trên, $D(G(z))$ đã xuống $6{,}2 \cdot 10^{-5}$ ở bước 500: mất mát gốc cho $G$ một gradient nhỏ hơn
khoảng 14.000 lần, và $G$ không bao giờ thoát ra (mục 7).

## 4 · Trò chơi một chiều với bộ phân biệt tốt nhất

Để thấy trò chơi thật sự cực tiểu cái gì, ta lấy một ví dụ nhỏ đến mức tính được mọi thứ chính xác, giống thẻ đầu của
demo. Dữ liệu là hai cụm Gauss, $p_{\text{data}} = \tfrac12\mathcal N(-2, 0{,}5^2) + \tfrac12\mathcal N(2, 0{,}5^2)$.
Bộ sinh chỉ có hai tham số: $G(z) = \mu + s\,z$ với $z \sim \mathcal N(0, 1)$, nên $p_g = \mathcal N(\mu, s^2)$.

Với $G$ cố định, $D$ cực đại $\mathbb E_x \log D(x) + \mathbb E_z \log(1 - D(G(z)))$ tại từng điểm $x$ một cách độc
lập, và lời giải là (Goodfellow và cộng sự, 2014, Mệnh đề 1):

$$D^*(x) = \frac{p_{\text{data}}(x)}{p_{\text{data}}(x) + p_g(x)}$$

Ô dưới viết $p_{\text{data}}$, $p_g$ và $D^*$ trong miền log, để không tràn số ở đuôi phân phối.

In [ ]:
MIX = ((-2.0, 0.5, 0.5), (2.0, 0.5, 0.5))  # (tâm, độ lệch chuẩn, trọng số) của hai cụm
assert [list(c) for c in MIX] == one["mix"]  # đúng dữ liệu của demo
LOG2PI = math.log(2 * math.pi)


def log_comps(x):
    """log(trọng số × mật độ) của từng cụm của p_data; mảng 2 × (cỡ của x)."""
    return np.stack([np.log(w) - 0.5 * ((x - m) / s) ** 2 - np.log(s) - 0.5 * LOG2PI for m, s, w in MIX])


def log_pdata(x):
    return np.logaddexp(*log_comps(x))


def log_pg(x, mu, s):
    return -0.5 * ((x - mu) / s) ** 2 - np.log(s) - 0.5 * LOG2PI


def d_star(x, mu, s):
    """D*(x) = p_data / (p_data + p_g)."""
    a = log_pdata(x)
    return np.exp(a - np.logaddexp(a, log_pg(x, mu, s)))


print(f"bộ sinh N(0,3; 0,5²): D*(−2) = {vn(d_star(-2.0, 0.3, 0.5), 4)} · D*(0,3) = {vn(d_star(0.3, 0.3, 0.5), 4)}"
      f" · D*(2) = {vn(d_star(2.0, 0.3, 0.5), 4)}")  # fmt: skip

### 4.1 · Một mạng thật học ra gần đúng $D^*$

$D^*$ là lời giải trên *mọi hàm*. Một mạng nhỏ học từ mẫu có đến gần nó không? Ô dưới viết một MLP ReLU tổng quát
(lan truyền ngược viết tay như Bài 06; mục 5 dùng lại nó cho cả $G$ lẫn $D$), rồi huấn luyện $D$ = 1 → 32 → 32 → 1
trong 2.000 bước, mỗi bước 256 mẫu thật và 256 mẫu của bộ sinh cố định $\mathcal N(0{,}3;\, 0{,}5^2)$.

Lớp `MLP` giữ R mạng độc lập cùng kiến trúc trong một lần: mỗi ma trận trọng số có thêm một chiều đầu R. Ở đây R = 1;
mục 5 huấn luyện 12 GAN song song bằng cách đó.

In [ ]:
class MLP:
    """R mạng ReLU độc lập, cùng kiến trúc (sizes), chạy song song. Mọi tham số nằm trong một mảng phẳng self.flat
    (W₀, b₀, W₁, b₁, … là các khung nhìn vào nó), nên Adam chỉ là vài phép tính trên một mảng."""

    def __init__(self, sizes, R, rng):
        shapes = [s for a, b in itertools.pairwise(sizes) for s in ((R, a, b), (R, 1, b))]
        self.flat = np.zeros(sum(math.prod(s) for s in shapes))
        self.p, i = [], 0
        for s in shapes:
            self.p.append(self.flat[i : i + math.prod(s)].reshape(s))
            i += math.prod(s)
        for l, (a, b) in enumerate(itertools.pairwise(sizes)):  # He: hệ số 2 cho lớp ẩn, 1 cho lớp ra; b = 0
            self.p[2 * l][...] = rng.standard_normal((R, a, b)) * np.sqrt((1 if l == len(sizes) - 2 else 2) / a)
        self.m, self.v, self.t = np.zeros_like(self.flat), np.zeros_like(self.flat), 0

    def fwd(self, x):
        """x: R × B × số đầu vào. Trả về đầu vào của từng lớp và cuối cùng là đầu ra (lớp cuối tuyến tính)."""
        hs = [x]
        for l in range(0, len(self.p), 2):
            u = hs[-1] @ self.p[l] + self.p[l + 1]  # quy ước hàng: h = f(xW + b)
            hs.append(np.maximum(u, 0) if l < len(self.p) - 2 else u)
        return hs

    def back(self, hs, d, params=True):
        """Lan truyền ngược d = ∂L/∂(đầu ra): gradient phẳng theo mọi tham số (nếu params) và ∂L/∂(đầu vào)."""
        g = []
        for l in range(len(self.p) - 2, -1, -2):
            if params:
                g += [d.sum(1, keepdims=True), hs[l // 2].transpose(0, 2, 1) @ d]  # ∂L/∂b, ∂L/∂W
            d = d @ self.p[l].transpose(0, 2, 1)
            if l > 0:
                d = d * (hs[l // 2] > 0)  # mặt nạ ReLU của lớp bên dưới
        return (np.concatenate([q.ravel() for q in g[::-1]]) if params else None), d

    def adam(self, g, lr, b1=0.5, b2=0.999):
        """Adam (Bài 07). β₁ = 0,5 như DCGAN và mọi lần chạy GAN của demo."""
        self.t += 1
        self.m += (1 - b1) * (g - self.m)
        self.v += (1 - b2) * (g * g - self.v)
        self.flat -= lr * (self.m / (1 - b1**self.t)) / (np.sqrt(self.v / (1 - b2**self.t)) + 1e-8)

In [ ]:
MU0, S0 = 0.3, 0.5  # bộ sinh cố định G(z) = 0,3 + 0,5·z
rng = np.random.default_rng(0)
D1 = MLP([1, 32, 32, 1], 1, rng)
B = 256
for step in range(2000):
    xr = np.where(rng.random(B) < 0.5, -2.0, 2.0) + 0.5 * rng.standard_normal(B)  # mẫu thật
    xf = MU0 + S0 * rng.standard_normal(B)  # mẫu của G
    hs = D1.fwd(np.r_[xr, xf][None, :, None])
    a = hs[-1][0, :, 0]
    da = np.r_[-sigmoid(-a[:B]), sigmoid(a[B:])] / B  # ∂L_D/∂a của cross-entropy: nhãn 1 thật, 0 giả
    D1.adam(D1.back(hs, da[None, :, None])[0], 3e-3, b1=0.9)  # D học một mình: Adam mặc định

xs = np.linspace(-4, 4, 801)
D_learn = sigmoid(D1.fwd(xs[None, :, None])[-1][0, :, 0])
D_opt = d_star(xs, MU0, S0)
wt = np.exp(log_pdata(xs)) + np.exp(log_pg(xs, MU0, S0))
wt /= wt.sum()  # trọng số: nơi có mẫu thật hoặc mẫu giả
err = (wt * np.abs(D_learn - D_opt)).sum()
half = xs[:-1][np.diff(np.sign(D_opt - 0.5)) != 0]
print(f"|D học được − D*| trung bình (nơi có mẫu): {vn(err, 4)}")
print("D* = ½ chỉ tại x ≈", " và ".join(vn(v, 2) for v in half), "(nơi p_data = p_g)")
assert err < 0.03 and len(half) == 2

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.fill_between(xs, np.exp(log_pdata(xs)), color=BLUE, alpha=0.35, label="p_data (dữ liệu)")
ax.plot(xs, np.exp(log_pg(xs, MU0, S0)), color=ORANGE, lw=2.2, label="p_g = N(0,3; 0,5²)")
ax2 = ax.twinx()
ax2.plot(xs, D_opt, color="black", lw=1.6, ls="--", label="D*(x), trục phải")
ax2.plot(xs, D_learn, color=GREEN, lw=2, ls=":", label="D học từ mẫu, trục phải")
ax2.axhline(0.5, color=GREY, lw=0.8)
ax2.set(ylim=(0, 1.02), ylabel="xác suất là thật")
ax2.spines["right"].set_visible(True)
ax.set(xlabel="x", ylabel="mật độ", ylim=(0, 0.85), xlim=(-4, 4))
ax.set_title("D tốt nhất với một bộ sinh cố định, và D mà một MLP học được")
fig.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=4)
vn_axes(fig)
plt.show()

Vùng xanh là dữ liệu, đường cam là bộ sinh. Đường đứt đen là $D^*$: gần 1 ở hai cụm (chỉ có dữ liệu), gần 0 ở giữa
(chỉ có mẫu giả). Đường chấm xanh lá là $D$ mà MLP học được từ mẫu; nó bám sát $D^*$ ở mọi nơi có mẫu. Lưu ý
$D^* = \tfrac12$ chỉ tại hai điểm, nơi hai mật độ cắt nhau: "$D = \tfrac12$" là tín hiệu của thành công chỉ khi nó đúng
**ở mọi nơi**, tức $p_g = p_{\text{data}}$. Mục 8 cho thấy vì sao nhìn $D$ trung bình trên các mẫu là không đủ.

### 4.2 · Trò chơi cực tiểu JSD

Thế $D^*$ vào giá trị của trò chơi $V(D, G) = \mathbb E_x \log D(x) + \mathbb E_z \log(1 - D(G(z)))$ (Goodfellow và
cộng sự, 2014, Định lý 1):

$$V(D^*, G) = -\log 4 + 2\cdot \mathrm{JSD}(p_{\text{data}} \,\|\, p_g), \qquad
\mathrm{JSD}(p \,\|\, q) = \tfrac12 \mathrm{KL}(p \,\|\, m) + \tfrac12 \mathrm{KL}(q \,\|\, m),\;\; m = \tfrac{p + q}{2}$$

JSD nằm giữa 0 (hai phân phối trùng nhau) và $\ln 2$ (không chồng lên nhau chút nào). Ô dưới tính JSD, KL theo chiều
$\mathrm{KL}(p_g \,\|\, p_{\text{data}})$ (mục 4.3 cần nó) và $V(D^*, G)$ bằng tích phân số trên một lưới mịn, cho vài
bộ sinh.

In [ ]:
X = np.linspace(-16, 16, 64001)  # lưới tích phân, như gan-1d.json
DX = X[1] - X[0]
A_X = log_pdata(X)


def divergences(mu, s):
    """JSD(p_data ‖ p_g), KL(p_g ‖ p_data) (nat) và V(D*, G), bằng tổng Riemann trên lưới X."""
    b = log_pg(X, mu, s)
    l = np.logaddexp(A_X, b)  # log(p_data + p_g)
    pa, pb = np.exp(A_X), np.exp(b)
    jsd = 0.5 * (pa * (A_X - l + np.log(2)) + pb * (b - l + np.log(2))).sum() * DX
    kl = (pb * (b - A_X)).sum() * DX
    v = ((pa * (A_X - l)).sum() + (pb * (b - l)).sum()) * DX  # E_data log D* + E_g log(1 − D*)
    return jsd, kl, v


print(f"{'bộ sinh':>16} {'JSD':>8} {'V(D*, G)':>10} {'−log 4 + 2·JSD':>16}")
for mu, s in [(0.3, 0.5), (2.0, 0.5), (0.0, 2.04), (6.0, 0.5)]:
    j, _, v = divergences(mu, s)
    print(f"  N({vn(mu, 1)}; {vn(s, 2)}²) {vn(j, 4):>8} {vn(v, 4):>10} {vn(-math.log(4) + 2 * j, 4):>16}")
    assert abs(v - (-math.log(4) + 2 * j)) < 1e-6
print(f"ln 2 = {vn(math.log(2), 4)}: JSD lớn nhất, khi hai phân phối không chồng lên nhau")

Ở mọi dòng, $V(D^*, G)$ tính trực tiếp bằng đúng $-\log 4 + 2\cdot\mathrm{JSD}$. Bộ sinh $\mathcal N(6;\, 0{,}5^2)$ gần
như không chồng lên dữ liệu, nên JSD gần chạm $\ln 2$.

### 4.3 · Mỗi mất mát đưa bộ sinh đi đâu?

Trong một bước huấn luyện, $G$ cập nhật với $D$ **giữ cố định**. Giả sử $D = D^*$ của bộ sinh hiện tại. Gradient theo
$(\mu, s)$ của hai mất mát đi qua $x = \mu + s\,z$ bằng quy tắc dây chuyền: $\partial/\partial\mu = \partial/\partial x$
và $\partial/\partial s = z\,\partial/\partial x$. Kỳ vọng theo $z \sim \mathcal N(0, 1)$ lấy bằng quy tắc hình thang
trên $[-8, 8]$ với bước 0,05, như demo. Hai đẳng thức cần kiểm tra:

- mất mát gốc: $\nabla\, \mathbb E_z \log(1 - D^*(G(z))) = \nabla\, 2\cdot\mathrm{JSD}$ (Goodfellow và cộng sự, 2014);
- không bão hòa: $\nabla\, \mathbb E_z[-\log D^*(G(z))] = \nabla\,[\mathrm{KL}(p_g \,\|\, p_{\text{data}}) - 2\cdot
  \mathrm{JSD}]$ (Arjovsky & Bottou, 2017, Định lý 2.5).

Ô dưới viết gradient giải tích của cả hai mất mát, rồi so với sai phân hữu hạn của JSD và KL ở bốn điểm $(\mu, s)$,
đúng bốn điểm mà `reel/gan/oned.py` của demo kiểm tra.

In [ ]:
Z = np.arange(-8, 8 + 1e-9, 0.05)  # nút tích phân cho z ~ N(0, 1)
WZ = np.exp(-0.5 * Z**2)
WZ /= WZ.sum()


def gen_grads(mu, s):
    """Gradient theo (μ, s) của hai mất mát của G, với D* giữ cố định tại bộ sinh hiện tại; nhận mảng (μ, s)."""
    mu, s = np.asarray(mu, float)[..., None], np.asarray(s, float)[..., None]
    x = mu + s * Z
    c = log_comps(x)
    a, b = np.logaddexp(*c), log_pg(x, mu, s)
    D = np.exp(a - np.logaddexp(a, b))  # D*(G(z))
    dA = (np.exp(c - a) * np.stack([-(x - m) / sd**2 for m, sd, _ in MIX])).sum(0)  # ∂/∂x log p_data
    dB = -(x - mu) / s**2  # ∂/∂x log p_g, với (μ, s) của D* giữ cố định
    dl = D * dA + (1 - D) * dB  # ∂/∂x log(p_data + p_g)
    out = {}
    for k, dx in (("sat", dB - dl), ("ns", -(dA - dl))):  # ∂/∂x của log(1 − D*) và của −log D*
        out[k] = np.stack([(WZ * dx).sum(-1), (WZ * dx * Z).sum(-1)], -1)
    out["eD"] = (WZ * D).sum(-1)  # trung bình D*(G(z))
    return out


worst = 0.0
for mu, s in [(1.0, 0.5), (0.342, 1.674), (3.0, 1.0), (-1.2, 0.8)]:
    g, h = gen_grads(mu, s), 1e-5
    dj, dk = [], []
    for dm, ds in ((h, 0), (0, h)):
        jp, kp, _ = divergences(mu + dm, s + ds)
        jm, km, _ = divergences(mu - dm, s - ds)
        dj.append((jp - jm) / (2 * h))
        dk.append((kp - km) / (2 * h))
    dj, dk = np.array(dj), np.array(dk)
    worst = max(worst, np.abs(g["sat"] - 2 * dj).max(), np.abs(g["ns"] - (dk - 2 * dj)).max())
    print(f"(μ, s) = ({vn(mu, 3)}; {vn(s, 3)}): ∂/∂μ gốc {vn(g['sat'][0], 5)} = 2·∂JSD {vn(2 * dj[0], 5)} · "
          f"không bão hòa {vn(g['ns'][0], 5)} = ∂(KL − 2·JSD) {vn(dk[0] - 2 * dj[0], 5)}")  # fmt: skip
print(f"lệch lớn nhất giữa gradient giải tích và sai phân hữu hạn: {sci(worst)}")
assert worst < 1e-5, "hai đẳng thức không khớp: hãy kiểm tra lại gen_grads"

Hai đẳng thức đúng tới sai số của sai phân hữu hạn. Vậy với $D$ tối ưu, mất mát gốc là gradient descent trên JSD,
đúng như lý thuyết hứa. Mất mát không bão hòa thì **không** cực tiểu JSD: nó đi theo
$\mathrm{KL}(p_g \,\|\, p_{\text{data}}) - 2\cdot\mathrm{JSD}$. KL theo chiều này (KL "ngược") phạt rất nặng việc đặt
xác suất vào nơi không có dữ liệu, nhưng gần như không phạt việc bỏ sót một phần dữ liệu (Arjovsky & Bottou, 2017).

Ô dưới cho thấy giá phải trả của mất mát gốc ở xa dữ liệu: với bộ sinh $\mathcal N(6;\, 0{,}5^2)$, $D^*$ gần như
chắc chắn mọi mẫu là giả, và gradient theo $\mu$ của mất mát gốc nhỏ hơn hàng chục nghìn lần.

In [ ]:
g_far = gen_grads(6.0, 0.5)
print(f"N(6; 0,5²): trung bình D*(G(z)) = {sci(g_far['eD'])} · ∂/∂μ: mất mát gốc {sci(g_far['sat'][0])}, "
      f"không bão hòa {vn(g_far['ns'][0], 2)} → gấp {vn(g_far['ns'][0] / g_far['sat'][0], 0)} lần")  # fmt: skip
grid_far = next(r for r in one["grid"] if r["mu"] == 6 and r["s"] == 0.5)
assert abs(g_far["sat"][0] / grid_far["gSat"] - 1) < 1e-3 and abs(g_far["ns"][0] / grid_far["gNs"] - 1) < 1e-3

### 🤔 Dự đoán trước
Bắt đầu từ bộ sinh $\mathcal N(0{,}3;\, 0{,}5^2)$ (hình ở mục 4.1), chạy gradient descent trên $(\mu, s)$ với từng mất
mát, mỗi bước tính lại $D^*$. Hai mất mát có đưa bộ sinh tới **cùng một chỗ** không? Nếu không, bên nào phủ được cả
hai cụm?

In [ ]:
STARTS = [(0.3, 0.5), (0.3, 1.5), (-1.0, 0.3), (1.0, 2.0), (4.0, 0.5), (5.0, 1.0)]  # như gan-1d.json


def flows(starts, steps=4000, lr=0.05, every=40):
    """Dòng gradient của cả hai mất mát từ mỗi điểm xuất phát, cùng lúc (D* tính lại ở mỗi bước, s ≥ 0,05)."""
    st = np.array(starts, float)
    mu, s = np.r_[st[:, 0], st[:, 0]], np.r_[st[:, 1], st[:, 1]]
    sat = np.arange(len(mu)) < len(st)  # nửa đầu: mất mát gốc; nửa sau: không bão hòa
    path = [np.c_[mu, s]]
    for t in range(1, steps + 1):
        g = gen_grads(mu, s)
        gr = np.where(sat[:, None], g["sat"], g["ns"])
        mu, s = mu - lr * gr[:, 0], np.maximum(0.05, s - lr * gr[:, 1])
        if t % every == 0:
            path.append(np.c_[mu, s])
    path = np.stack(path, 1)  # (2·số điểm) × số mốc × 2
    return path[: len(st)], path[len(st) :]


path_sat, path_ns = flows(STARTS)
print(f"{'xuất phát':>14} │ {'mất mát gốc → (μ, s), JSD':>28} │ {'không bão hòa → (μ, s), JSD':>28}")
for i, (m0, s0) in enumerate(STARTS):
    row = []
    for path, key in ((path_sat, "sat"), (path_ns, "ns")):
        mu, s = path[i, -1]
        row.append(f"({vn(mu, 2)}; {vn(s, 2)}), {vn(divergences(mu, s)[0], 4)}")
        assert np.allclose([mu, s], one["flows"][i][key]["end"], atol=2e-3), "khác với demo"
    print(f"  ({vn(m0, 1)}; {vn(s0, 1)}) │ {row[0]:>28} │ {row[1]:>28}")

Không. Từ $\mathcal N(0{,}3;\, 0{,}5^2)$, **mất mát gốc** trải bộ sinh thành một Gauss rộng $\mathcal N(0;\, 2{,}04^2)$
phủ cả hai cụm, đúng Gauss đơn có JSD nhỏ nhất. **Mất mát không bão hòa** chọn hẳn một cụm, $\mathcal N(2;\, 0{,}5^2)$.
Theo JSD, Gauss rộng tốt hơn (0,2010 so với 0,2157). Nhưng nó đặt nhiều xác suất vào khoảng trống giữa hai cụm, nơi
không có dữ liệu, và đó đúng là thứ mà KL ngược phạt nặng. Ô dưới in hai đại lượng tại hai điểm đến.

Từ $(0{,}3;\, 1{,}5)$, mất mát không bão hòa cũng dừng ở một Gauss rộng, $\mathcal N(0;\, 1{,}71^2)$: đó là một cực
tiểu địa phương của $\mathrm{KL} - 2\cdot\mathrm{JSD}$, không phải điểm tốt nhất của nó.

In [ ]:
for name, (mu, s) in [("Gauss rộng của mất mát gốc", path_sat[0, -1]), ("một cụm", path_ns[0, -1]),
                      ("Gauss rộng của không bão hòa", path_ns[1, -1])]:  # fmt: skip
    j, k, _ = divergences(mu, s)
    print(f"{name:>29} N({vn(mu, 2)}; {vn(s, 2)}²): JSD {vn(j, 4)} · KL(p_g ‖ p_data) {vn(k, 3)} · "
          f"KL − 2·JSD {vn(k - 2 * j, 3)}")  # fmt: skip


def jsd_grid(mus, ss, n=1601):
    """JSD trên một lưới (μ, s), với lưới tích phân thô hơn (chỉ để vẽ đường đồng mức)."""
    x = np.linspace(-16, 16, n)
    a = log_pdata(x)
    b = log_pg(x, mus[..., None], ss[..., None])
    l = np.logaddexp(a, b) - np.log(2)
    return 0.5 * (np.exp(a) * (a - l) + np.exp(b) * (b - l)).sum(-1) * (x[1] - x[0])


M_, S_ = np.meshgrid(np.linspace(-4, 6, 81), np.linspace(0.2, 3, 57))
fig, ax = plt.subplots(figsize=(8, 4.4))
cs = ax.contour(M_, S_, jsd_grid(M_, S_), levels=[0.205, 0.22, 0.25, 0.3, 0.4, 0.5, 0.6, 0.68], colors=GREY,
                linewidths=0.8)  # fmt: skip
ax.clabel(cs, fmt=lambda v: vn(v, 3), fontsize=7)
for i, (m0, s0) in enumerate(STARTS):
    ax.plot(*path_sat[i].T, color=GREEN, lw=2)
    ax.plot(*path_ns[i].T, color=ORANGE, lw=2, ls="--")
    ax.plot(m0, s0, "o", ms=8, mfc="white", mec="black")
    ax.plot(*path_sat[i, -1], "o", ms=8, color=GREEN)
    ax.plot(*path_ns[i, -1], "s", ms=7, color=ORANGE)
ax.plot([], [], color=GREEN, lw=2, marker="o", label="mất mát gốc log(1 − D*) → cực tiểu JSD")
ax.plot([], [], color=ORANGE, lw=2, ls="--", marker="s", label="không bão hòa −log D* → KL(p_g ‖ p_data) − 2·JSD")
ax.plot([], [], "o", mfc="white", mec="black", label="điểm xuất phát")
ax.set(xlabel="μ (tâm của bộ sinh)", ylabel="s (độ rộng của bộ sinh)", xlim=(-4, 6), ylim=(0.2, 3))
ax.set_title("Dòng gradient với D* tính lại mỗi bước; đường xám: đồng mức JSD (nat)")
fig.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, 0.02), ncol=2)
vn_axes(fig)
plt.show()

Đường xám là các đồng mức của JSD trên mặt $(\mu, s)$; đáy của nó ở $(0;\, 2{,}04)$. Đường liền xanh lá (mất mát gốc)
từ bốn trong sáu điểm xuất phát chạy về đúng đáy đó; hai điểm còn lại bắt đầu gần một cụm và dừng ở cụm ấy, một cực
tiểu địa phương của JSD. Đường đứt cam (không bão hòa) kết thúc ở một cụm từ năm trong sáu điểm xuất phát (ô vuông ở
$\mu = \pm 2$, $s = 0{,}5$). Cả hai cách đều là "một Gauss tốt nhất" theo một nghĩa nào đó, nhưng mất mát mà mọi người dùng trong thực
tế thiên về **chọn một mode và vẽ nó thật sắc**, thay vì phủ hết dữ liệu. Đó là mầm của mode collapse.

Đây vẫn là trò chơi lý tưởng: $D$ luôn bằng $D^*$, bộ sinh chỉ có hai tham số. Mạng thật chỉ có một $D$ gần đúng, cập
nhật xen kẽ với $G$, và mọi chuyện phức tạp hơn.

## 5 · GAN hai chiều trên một vòng 8 cụm

Giờ cả $G$ lẫn $D$ đều là mạng thật, cập nhật xen kẽ, như thẻ thứ hai của demo. Dữ liệu là 8 cụm Gauss rất hẹp
(độ lệch chuẩn 0,02) đặt đều trên một vòng tròn bán kính 2, một bài thử kinh điển cho mode collapse (Metz và cộng
sự, 2016). Kiến trúc và công thức huấn luyện giống `viz/gan2d.ts` của demo:
- $G$: $z$ (16 chiều) → ReLU 32 → ReLU 32 → 2; $D$: $x$ (2 chiều) → ReLU 32 → ReLU 32 → 1 logit;
- mỗi bước: một bước cho $D$ trên một batch thật và một batch giả mới (64 mẫu mỗi loại), rồi một bước cho $G$ trên
  nhiễu mới, chống lại $D$ vừa cập nhật; Adam với $\eta = 0{,}002$, $\beta_1 = 0{,}5$ (demo lưu sẵn cả $\eta =
  0{,}001$ và $0{,}002$; với 4.000 bước, $0{,}002$ đi xa hơn).

Để đếm số cụm được phủ, ta dùng thước đo của VEEGAN (Srivastava và cộng sự, 2017), như demo: một mẫu là **tốt** nếu nó
cách cụm gần nhất không quá 3 độ lệch chuẩn; một cụm **có mẫu** nếu có ít nhất một mẫu tốt, và **đủ phần** nếu nó nhận
ít nhất ¼ phần công bằng của mình (1.000 mẫu / 8 cụm / 4 ≈ 31 mẫu). Mọi lần đo dùng cùng 1.000 nhiễu cố định.

In [ ]:
MODES = 2 * np.c_[np.cos(np.arange(8) * np.pi / 4), np.sin(np.arange(8) * np.pi / 4)]  # 8 tâm trên vòng bán kính 2
SD = 0.02


def ring_data(rng, R, B):
    """R × B mẫu thật: chọn một cụm đều nhau, cộng nhiễu Gauss độ lệch chuẩn 0,02."""
    return MODES[rng.integers(0, 8, (R, B))] + SD * rng.standard_normal((R, B, 2))


def score(x):
    """x: R × n × 2. Số mẫu tốt của từng cụm (R × 8) và tỉ lệ mẫu tốt của mỗi lần chạy."""
    d2 = ((x[:, :, None, :] - MODES) ** 2).sum(-1)
    k, good = d2.argmin(-1), np.sqrt(d2.min(-1)) <= 3 * SD
    counts = np.stack([np.bincount(k[r][good[r]], minlength=8) for r in range(len(x))])
    return counts, good.mean(1)


c, q = score(ring_data(np.random.default_rng(1), 1, 1000))
print(f"1.000 mẫu thật: {(c > 0).sum()} cụm có mẫu, {(c >= 1000 / 32).sum()} cụm đủ phần, {pct(q[0], 1)} mẫu tốt")

### Phạt gradient R1

Ta sẽ so bốn cấu hình: hai mất mát của $G$ (mục 3), mỗi mất mát có và không có **phạt gradient R1** (Mescheder và
cộng sự, 2018). R1 cộng vào mất mát của $D$ một số hạng phạt độ dốc của logit $a(x)$ **tại dữ liệu thật**:

$$R_1 = \frac{\gamma}{2}\, \mathbb E_{x \sim p_{\text{data}}} \big[\lVert \nabla_x a(x) \rVert^2\big]$$

Khi $G$ đã trùng với dữ liệu, một $D$ không bị phạt vẫn có thể dốc đứng quanh mỗi điểm thật và đẩy $G$ đi khỏi chỗ
đúng; $R_1$ ép $D$ phẳng ở đó, và Mescheder và cộng sự chứng minh nó làm huấn luyện hội tụ cục bộ, trong khi huấn
luyện không chính quy hóa có thể xoay vòng mãi quanh điểm cân bằng.

Với $D$ là một mạng ReLU, trong vùng mà các mặt nạ ReLU $m_1, m_2$ cố định thì $a(x)$ tuyến tính theo $x$, nên
$\nabla_x a = u_1 W_0^{\top}$ với $u_2 = m_2 \odot W_2^{\top}$ và $u_1 = m_1 \odot (u_2 W_1^{\top})$ (quy ước hàng).
Đạo hàm của $R_1$ theo các ma trận trọng số đi ngược lại đúng chuỗi đó; các độ lệch $b$ không nhận gradient nào.
Ô dưới viết $R_1$, rồi kiểm tra nó và lan truyền ngược của cả hai mạng bằng sai phân hữu hạn trên hai mạng tí hon.

In [ ]:
def r1_grad(D, hs, gamma):
    """R1 trên các mẫu thật đã chạy qua D (hs = D.fwd(x)), cho D = 2 → H → H → 1: gradient phẳng và giá trị.
    gamma: mảng R × 1 × 1 (một γ cho mỗi lần chạy)."""
    W0, W1, W2 = D.p[0], D.p[2], D.p[4]
    m1, m2 = hs[1] > 0, hs[2] > 0
    u2 = m2 * W2.transpose(0, 2, 1)  # R × B × H
    u1 = m1 * (u2 @ W1.transpose(0, 2, 1))
    gx = u1 @ W0.transpose(0, 2, 1)  # ∇ₓa(x): R × B × 2
    dgx = gamma * gx / gx.shape[1]  # ∂R1/∂(∇ₓa)
    e = m1 * (dgx @ W0)
    gW = [dgx.transpose(0, 2, 1) @ u1, e.transpose(0, 2, 1) @ u2, ((e @ W1) * m2).sum(1)[:, :, None]]
    flat = [gW[0], np.zeros_like(D.p[1]), gW[1], np.zeros_like(D.p[3]), gW[2], np.zeros_like(D.p[5])]
    return np.concatenate([q.ravel() for q in flat]), 0.5 * gamma[:, 0, 0] * (gx**2).sum(2).mean(1)


def fd_worst(net, f, g, rng, k=12, h=1e-6):
    """Sai số tương đối lớn nhất giữa gradient g và sai phân hữu hạn của f() trên k tham số ngẫu nhiên của net."""
    worst = 0.0
    for i in rng.choice(net.flat.size, k, replace=False):
        old = net.flat[i]
        net.flat[i] = old + h
        fp = f()
        net.flat[i] = old - h
        fm = f()
        net.flat[i] = old
        num = (fp - fm) / (2 * h)
        worst = max(worst, abs(num - g[i]) / max(1e-7, abs(num) + abs(g[i])))
    return worst


rng = np.random.default_rng(3)
Gt, Dt = MLP([3, 5, 4, 2], 1, rng), MLP([2, 6, 5, 1], 1, rng)
Dt.flat += 0.1 * rng.standard_normal(Dt.flat.size)  # độ lệch khác 0, để mặt nạ ReLU không tầm thường
xr_t, z_t, gam_t = rng.standard_normal((1, 7, 2)), rng.standard_normal((1, 7, 3)), np.full((1, 1, 1), 0.7)
gR1 = r1_grad(Dt, Dt.fwd(xr_t), gam_t)[0]
err_r1 = fd_worst(Dt, lambda: r1_grad(Dt, Dt.fwd(xr_t), gam_t)[1][0], gR1, rng)


def g_loss_ns():
    return softplus(-Dt.fwd(Gt.fwd(z_t)[-1])[-1]).mean()


hg = Gt.fwd(z_t)
hf = Dt.fwd(hg[-1])
_, dx = Dt.back(hf, -sigmoid(-hf[-1]) / 7, params=False)
err_g = fd_worst(Gt, g_loss_ns, Gt.back(hg, dx)[0], rng)
print(f"sai số tương đối lớn nhất: R1 theo trọng số của D {sci(err_r1)} · mất mát của G qua D {sci(err_g)}")
assert err_r1 < 1e-5 and err_g < 1e-5

In [ ]:
def train_ring(losses, gammas, steps, lr=2e-3, dz=16, H=32, B=64, seed=0, every=250):
    """Huấn luyện len(losses) GAN cùng lúc (mỗi lần chạy một mất mát 'ns'/'sat' và một γ). Mỗi `every` bước ghi lại
    số mẫu tốt của từng cụm trên 1.000 nhiễu cố định, cùng D(thật) và D(giả) trung bình của batch vừa học."""
    R = len(losses)
    rng = np.random.default_rng(seed)
    G, D = MLP([dz, H, H, 2], R, rng), MLP([2, H, H, 1], R, rng)
    ns = np.array([loss == "ns" for loss in losses], float)[:, None, None]
    gam = np.asarray(gammas, float)[:, None, None]
    z_eval = np.broadcast_to(np.random.default_rng(12345).standard_normal((1000, dz)), (R, 1000, dz))
    log = []
    for s in range(steps + 1):
        if s % every == 0:
            counts, good = score(G.fwd(z_eval)[-1])
            log.append({"step": s, "counts": counts, "good": good})
        if s == steps:
            break
        # bước của D: cross-entropy (thật nhãn 1, giả nhãn 0), cộng R1 trên mẫu thật
        xr, xf = ring_data(rng, R, B), G.fwd(rng.standard_normal((R, B, dz)))[-1]
        hs = D.fwd(np.concatenate([xr, xf], 1))
        a = hs[-1]
        gD, _ = D.back(hs, np.concatenate([-sigmoid(-a[:, :B]), sigmoid(a[:, B:])], 1) / B)
        if gam.any():
            gD += r1_grad(D, [h[:, :B] for h in hs], gam)[0]
        D.adam(gD, lr)
        # bước của G trên nhiễu mới, chống lại D vừa cập nhật; gradient đi xuyên qua D (∂/∂x) rồi vào G
        hg = G.fwd(rng.standard_normal((R, B, dz)))
        hf = D.fwd(hg[-1])
        af = hf[-1]
        _, dx = D.back(hf, -(ns * sigmoid(-af) + (1 - ns) * sigmoid(af)) / B, params=False)
        G.adam(G.back(hg, dx)[0], lr)
        if (s + 1) % every == 0:
            log.append({"dreal": sigmoid(a[:, :B]).mean((1, 2)), "dfake": sigmoid(a[:, B:]).mean((1, 2))})
    return G, D, [e for e in log if "step" in e], [e for e in log if "dreal" in e]


CONFIGS = [("ns", 0.0), ("sat", 0.0), ("ns", 0.1), ("sat", 0.1)]  # (mất mát của G, γ của R1)
NAMES = ["không bão hòa, không R1", "mất mát gốc, không R1", "không bão hòa, R1 γ = 0,1", "mất mát gốc, R1 γ = 0,1"]
SEEDS = 3

### 🤔 Dự đoán trước
Ô dưới huấn luyện 4 cấu hình × 3 lần chạy (12 GAN song song) trong 4.000 bước. Không có R1 và với mất mát không bão
hòa, bộ sinh phủ được bao nhiêu trong 8 cụm? Và ở cuối, $D(\text{thật})$ có gần ½ không?

In [ ]:
G2, D2, ring_log, d_log = train_ring([c[0] for c in CONFIGS for _ in range(SEEDS)],
                                     [c[1] for c in CONFIGS for _ in range(SEEDS)], 4000)  # fmt: skip
final = ring_log[-1]
modes, fair = (final["counts"] > 0).sum(1), (final["counts"] >= 1000 / 32).sum(1)
dreal = np.mean([e["dreal"] for e in d_log[-4:]], 0)  # trung bình 1.000 bước cuối
dfake = np.mean([e["dfake"] for e in d_log[-4:]], 0)
for i, name in enumerate(NAMES):
    r = slice(SEEDS * i, SEEDS * (i + 1))
    print(f"{name:>26}: cụm có mẫu {modes[r].tolist()} · đủ phần {fair[r].tolist()} · "
          f"mẫu tốt {span(final['good'][r] * 100, 0)}% · D(thật) {span(dreal[r])} · D(giả) {span(dfake[r])}")  # fmt: skip
# Kiểm tra bức tranh chung chứ không phải từng con số: GAN khuếch đại sai số làm tròn, nên trên CPU khác (Colab là x86)
# từng lần chạy có thể đi đường khác. Các ngưỡng dưới đã thử với 4 seed × float32/float64.
no_r1, with_r1, sat_no = slice(0, 2 * SEEDS), slice(2 * SEEDS, 4 * SEEDS), slice(SEEDS, 2 * SEEDS)
assert modes[no_r1].mean() <= 4 and modes[with_r1].mean() >= 6.5, "R1 phải phủ nhiều cụm hơn hẳn"
assert final["good"][sat_no].mean() < 0.3 and fair[sat_no].max() <= 2, "mất mát gốc không R1 phải kẹt"
assert dreal[no_r1].mean() > 0.7 and abs(dreal[with_r1] - 0.5).max() < 0.08, "D thắng khi không có R1, ≈ ½ khi có"

Hình dưới vẽ lần chạy đầu tiên của mỗi cấu hình. Hàng trên: 1.000 mẫu của bộ sinh ở bước cuối (ô vuông cam) trên dữ
liệu thật (chấm xanh). Hàng dưới: theo thời gian (trục ngang), mỗi hàng là một cụm, ô càng đậm là cụm đó càng nhiều
mẫu tốt.

In [ ]:
real_pts = ring_data(np.random.default_rng(7), 1, 300)[0]
x_end = G2.fwd(np.random.default_rng(12345).standard_normal((G2.p[0].shape[0], 1000, 16)))[-1]
steps_logged = np.array([e["step"] for e in ring_log])
counts_t = np.stack([e["counts"] for e in ring_log], 2)  # lần chạy × cụm × mốc
fig, axes = plt.subplots(2, 4, figsize=(10, 5.2), gridspec_kw={"height_ratios": [1.25, 1]})
for i, name in enumerate(NAMES):
    r = SEEDS * i  # lần chạy đầu tiên của cấu hình i
    ax = axes[0, i]
    ax.scatter(*real_pts.T, s=6, color=BLUE, marker="o", label="dữ liệu thật")
    ax.scatter(*x_end[r].T, s=4, color=ORANGE, marker="s", alpha=0.5, label="mẫu của G")
    ax.set(xlim=(-3, 3), ylim=(-3, 3), aspect="equal", xticks=[-2, 0, 2], yticks=[-2, 0, 2])
    ax.set_title(f"{name.replace(', ', chr(10))}\n{modes[r]} cụm có mẫu, {fair[r]} đủ phần", fontsize=9)
    ax = axes[1, i]
    ax.imshow(counts_t[r] / 1000, aspect="auto", cmap="Oranges", vmin=0, vmax=0.25, interpolation="nearest",
              extent=[0, steps_logged[-1], 7.5, -0.5])  # fmt: skip
    ax.set(xlabel="bước", yticks=range(8), yticklabels=[str(k) for k in range(8)] if i == 0 else [])
    ax.set_xticks([0, 2000, 4000])
axes[1, 0].set_ylabel("cụm")
axes[0, 0].legend(frameon=False, fontsize=7, loc="lower left", markerscale=2)
plt.tight_layout()
vn_axes(fig)
plt.show()

Không có R1, bộ sinh với mất mát không bão hòa nhanh chóng dồn mẫu vào 1–3 cụm và ở yên đó: các hàng còn lại của
hình dưới gần như trắng suốt quá trình. Đó là **mode collapse**: nhiều $z$ khác nhau cho ra cùng một chỗ. Đúng như
trò chơi một chiều ở mục 4: mất mát không bão hòa chấp nhận bỏ sót cụm, miễn là mẫu nào cũng trông thật.

Mất mát gốc không R1 còn tệ hơn: gần như không có mẫu nào rơi đúng vào một cụm. $D$ thắng quá dễ (D(thật) khoảng
0,9, D(giả) khoảng 0,1 ở các lần chạy trên), và như mục 3, gradient của mất mát gốc tắt khi $D$ bác bỏ dễ dàng.

Với R1, cả hai mất mát đều trải mẫu ra khắp vòng, và $D(\text{thật}) \approx D(\text{giả}) \approx \tfrac12$. Sau
4.000 bước, mẫu chưa gọn vào từng cụm (nhiều mẫu nằm trên cung giữa hai cụm), nên số cụm "đủ phần" còn thấp; các lần
chạy dài hơn của demo cho thấy điều gì xảy ra tiếp.

### So với các lần chạy lưu sẵn của demo
`gan-2d.json` lưu cùng công thức này, chạy bằng mã TypeScript của demo (`viz/gan2d.ts`) với 5 seed và 20.000 bước, số
cụm đo mỗi 500 bước trên 2.000 mẫu. Số ngẫu nhiên khác hẳn của ta, nên chỉ so được về chất. Ô dưới đọc số cụm ở bước
4.000 (để so với lần chạy của ta) và ở bước 20.000.

In [ ]:
def checkpoints(run):
    """Các mốc (mỗi 500 bước) của một lần chạy lưu sẵn: cụm có mẫu, cụm đủ phần, tỉ lệ mẫu tốt, cụm đông nhất."""
    b = np.frombuffer(base64.b64decode(run["traj"]), np.uint8).reshape(-1, 4).astype(int)
    return {"modes": b[:, 0], "fair": b[:, 1], "good": b[:, 2] / 250, "top": b[:, 3] - 1}


at4k = 4000 // two["setup"]["every"]
for (loss, g), name in zip(CONFIGS, NAMES):
    runs = two["runs"][f"ring-H32-{loss}-dz16-r{g:g}-alt-lr0.002"]["seeds"]
    cp = [checkpoints(r) for r in runs]
    print(f"{name:>26} (5 seed): bước 4.000: cụm có mẫu {[int(c['modes'][at4k]) for c in cp]} · "
          f"bước 20.000: có mẫu {[r['final']['modes'] for r in runs]}, đủ phần {[r['final']['fair'] for r in runs]}, "
          f"D(thật) {span([r['final']['dReal'] for r in runs])}")  # fmt: skip

Cùng bức tranh: ở bước 4.000, không R1 thì 1–5 cụm có mẫu, có R1 thì cả 8 cụm ở mọi seed. Sau 20.000 bước, R1 gom
mẫu vào từng cụm: với mất mát không bão hòa, 7–8 cụm đủ phần ở cả 5 seed (mất mát gốc: 3–8). Không R1 thì sau
20.000 bước vẫn chỉ 1–4 cụm có mẫu. Thêm bước không chữa được mode collapse.

### Nhảy cụm, hay chỉ là nhiễu của thước đo?
Metz và cộng sự (2016) mô tả một kiểu collapse không đứng yên: bộ sinh dồn vào một cụm, $D$ học bác bỏ cụm đó, bộ sinh
nhảy sang cụm khác, và vòng đuổi không dừng. Một cách đo là đếm số lần "cụm đông nhất" đổi chỗ. Nhưng thước đo đó chỉ
có nghĩa khi có đủ mẫu tốt để đếm. Ô dưới xét các lần chạy lưu sẵn với mất mát gốc và lớp ẩn 64, nơi "cụm đông nhất"
đổi chỗ nhiều nhất trong nửa sau của huấn luyện.

In [ ]:
hop = two["runs"]["ring-H64-sat-dz16-r0-alt-lr0.001"]["seeds"]
for r in hop:
    cp = checkpoints(r)
    half = slice(len(cp["top"]) // 2, None)
    top = cp["top"][half][cp["top"][half] >= 0]
    print(f"seed {r['seed']}: cụm đông nhất đổi {int((top[1:] != top[:-1]).sum())} lần, qua {len(set(top))} cụm · "
          f"nhưng mẫu tốt nhiều nhất chỉ {pct(cp['good'][half].max(), 1)} của 2.000 mẫu")  # fmt: skip
pts = np.frombuffer(base64.b64decode(hop[0]["pts"]), "<i2").reshape(-1, 2) / 1000  # 300 mẫu cuối của seed 0
d_mode = np.sqrt(((pts[:, None] - MODES) ** 2).sum(-1)).min(1)
print(f"seed 0, 300 mẫu cuối: khoảng cách trung vị tới cụm gần nhất {vn(np.median(d_mode), 2)}, tức "
      f"{vn(np.median(d_mode) / SD, 0)} độ lệch chuẩn của một cụm; bán kính từ {vn(np.hypot(*pts.T).min(), 1)} "
      f"đến {vn(np.hypot(*pts.T).max(), 1)}")  # fmt: skip

Ở cả 3 seed, "cụm đông nhất" đổi chỗ 15–16 lần và đi qua 7 cụm. Nhưng gần như không có mẫu nào nằm trong một cụm:
nhiều nhất 0,4–0,8% của 2.000 mẫu, tức 8–16 mẫu chia cho 8 cụm, nên "cụm đông nhất" đổi chỗ theo nhiễu. 300 mẫu cuối
của seed 0 nằm rải trên một vùng rộng, trung vị cách cụm gần nhất khoảng 50 độ lệch chuẩn. Đó không phải bộ sinh nhảy
từ cụm này sang cụm khác, mà là một bộ sinh đã hỏng hẳn (mất mát gốc, $D$ thắng tuyệt đối).

Còn những lần chạy collapse thật thì sao? Ô dưới chọn mọi lần chạy lưu sẵn trên vòng 8 cụm mà ở ít nhất 70% số mốc của
nửa sau, ít nhất 30% mẫu là mẫu tốt và chỉ 1–3 cụm đủ phần, rồi đếm cụm đông nhất của chúng đi qua bao nhiêu cụm.

In [ ]:
visited = []
for run in two["runs"].values():
    for r in run["seeds"] if run["cfg"]["data"] == "ring" else []:
        cp = checkpoints(r)
        half = slice(len(cp["top"]) // 2, None)
        ok = (cp["good"][half] >= 0.3) & (cp["fair"][half] <= 3)
        if r["final"] and ok.mean() >= 0.7:
            visited.append(len(set(cp["top"][half][ok])))
print(f"{len(visited)} lần chạy collapse thật; cụm đông nhất trong nửa sau đi qua: "
      f"1 cụm ở {visited.count(1)} lần, 2 cụm ở {visited.count(2)} lần, nhiều hơn ở {sum(v > 2 for v in visited)} lần")  # fmt: skip

Ở mọi lần chạy collapse thật, cụm đông nhất đứng yên hoặc chỉ đổi qua lại giữa 2 cụm. Trong các lần chạy này, bộ sinh
không "nhảy vòng" qua các cụm như Metz và cộng sự (2016) mô tả; nó dừng ở một hai cụm và ở yên đó.

Bài học chung: một thước đo chỉ đáng tin khi điều kiện của nó được thỏa mãn. Mục 8 gặp lại chuyện này với $D \approx
\tfrac12$.

## 6 · Chữ số từ nhiễu: bộ sinh MNIST của demo

Trên MNIST, $G$ có đúng kiến trúc bộ giải mã của Bài 19: $z$ (16 chiều) → ReLU 128 → ReLU 256 → 784 → sigmoid, và
$D$ đối xứng với bộ mã hóa: 784 → LeakyReLU 256 → LeakyReLU 128 → 1 logit. Hai mạng học bằng đúng vòng lặp của mục 5
(Adam $\beta_1 = 0{,}5$, batch 64, 20.000 bước) trên 55.000 ảnh xám. Huấn luyện một lần mất nhiều phút trên CPU,
nên ta nạp bộ sinh tốt nhất của lưới (mất mát không bão hòa, R1 γ = 10, η = 0,001, seed 0) từ `gan-g16.json`, cùng
định dạng với Bài 19: lớp ẩn float16, ma trận lớp ra int8 với một hệ số float16 cho mỗi cột.

Ô dưới giải nén nó, sinh 10.000 chữ số từ nhiễu `default_rng(7)` (đúng dòng nhiễu mà các phép đo của demo dùng), rồi
so 24 ảnh đầu với 24 ảnh mà demo lưu (sinh bằng trọng số chưa làm tròn).

In [ ]:
def b64(s, dtype):
    """Giải mã một mảng mà demo lưu dạng base64 (little-endian)."""
    return np.frombuffer(base64.b64decode(s), dtype)


def unpack(t):
    if "f16" in t:
        return b64(t["f16"], "<f2").astype(np.float32).reshape(t["shape"])
    scale = b64(t["scale"], "<f2").astype(np.float32)  # một hệ số cho mỗi cột
    return b64(t["i8"], np.int8).astype(np.float32).reshape(t["shape"]) * scale


Gm = {k: unpack(t) for k, t in g16["dec"].items()}


def generate(z):
    """G(z): 16 → ReLU 128 → ReLU 256 → 784 → sigmoid; mỗi hàng của kết quả là một ảnh, điểm ảnh trong [0, 1]."""
    h1 = np.maximum(z @ Gm["d0.W"] + Gm["d0.b"], 0)
    h2 = np.maximum(h1 @ Gm["d1.W"] + Gm["d1.b"], 0)
    return sigmoid(h2 @ Gm["out.W"] + Gm["out.b"])


def u8(s):
    """Ảnh mà demo lưu dạng uint8 base64 → mảng n × 784 trong [0, 1]."""
    return b64(s, np.uint8).reshape(-1, 784).astype(np.float32) / 255


Xg = generate(np.random.default_rng(7).standard_normal((10000, 16)).astype(np.float32))
n_par = sum(v.size for v in Gm.values())
diff = np.abs(Xg[:24] - u8(mn["images"]["ns-r10-lr0.001"]))
print({k: v.shape for k, v in Gm.items() if k.endswith(".W")}, f"· {vn(n_par, 0)} tham số")
print(
    f"24 ảnh đầu so với ảnh demo lưu: lệch trung bình {vn(diff.mean(), 4)}, lớn nhất {vn(diff.max(), 3)} (trên thang 0–1)"
)
assert n_par == 236688 and diff.mean() < 0.002 and diff.max() < 0.06, "khác với demo: hãy kiểm tra lại bộ sinh"

Lệch lớn nhất vài phần trăm của thang điểm ảnh: đó là phần làm tròn int8/float16 cộng với làm tròn ảnh lưu về
uint8. Hình dưới là 48 mẫu đầu tiên, và hàng cuối là 9 ảnh trên đường thẳng trong không gian nhiễu từ mẫu 1 đến mẫu 2
(hai ảnh đầu ở góc trên bên trái).

In [ ]:
z_ab = np.random.default_rng(7).standard_normal((2, 16)).astype(np.float32)
frames = generate(z_ab[0] + np.linspace(0, 1, 9)[:, None] * (z_ab[1] - z_ab[0]))
fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 5.6), gridspec_kw={"height_ratios": [4, 1.1]})
a1.imshow(tile(Xg[:48], 4, 12), cmap="gray_r", vmin=0, vmax=1)
a1.set_title("48 chữ số do bộ sinh tạo từ nhiễu z ~ N(0, I) (R1 γ = 10, seed 0)", fontsize=10)
a2.imshow(tile(frames, 1, 9), cmap="gray_r", vmin=0, vmax=1)
a2.set_title("nội suy thẳng trong không gian nhiễu, từ mẫu 1 đến mẫu 2", fontsize=10)
for ax in (a1, a2):
    ax.axis("off")
plt.tight_layout()
plt.show()

Phần lớn mẫu đọc được thành chữ số, nét liền và sắc hơn mẫu của VAE ở Bài 19, dù một số ảnh còn méo. Đi trên đường
thẳng giữa hai nhiễu, ảnh biến đổi liên tục từ chữ số này sang chữ số kia: bộ sinh là một hàm trơn của $z$.

### 🤔 Dự đoán trước
Bộ sinh **không bao giờ** nhận một ảnh thật: mọi thông tin về dữ liệu đến với nó qua gradient của $D$ (Goodfellow và
cộng sự, 2014, mục 6). Vậy mẫu của nó có thể là bản sao của ảnh huấn luyện không? Đo bằng khoảng cách điểm ảnh
(chuẩn Euclid trên 784 điểm) từ mỗi mẫu tới ảnh huấn luyện gần nhất: mẫu của GAN gần ảnh huấn luyện **hơn** hay **xa
hơn** một ảnh kiểm tra thật (một chữ số mà mô hình chưa thấy)?

In [ ]:
def nearest(A, T, bs=250):
    """Khoảng cách điểm ảnh từ mỗi hàng của A tới hàng gần nhất của T, và chỉ số của hàng đó."""
    tn = (T.astype(np.float64) ** 2).sum(1)
    dist, idx = np.empty(len(A)), np.empty(len(A), np.int64)
    for i in range(0, len(A), bs):
        x = A[i : i + bs]
        d2 = (x.astype(np.float64) ** 2).sum(1)[:, None] + tn - 2 * (x @ T.T).astype(np.float64)
        idx[i : i + bs] = d2.argmin(1)
        dist[i : i + bs] = np.sqrt(np.maximum(d2[np.arange(len(x)), idx[i : i + bs]], 0))
    return dist, idx


d_gan, i_gan = nearest(Xg[:1000], Xtr)  # 1.000 mẫu → 55.000 ảnh huấn luyện
d_test, _ = nearest(Xte[:1000], Xtr)  # 1.000 ảnh kiểm tra thật → 55.000 ảnh huấn luyện
QS = (0.01, 0.1, 0.5)
thr = np.quantile(d_test, 0.01)  # "sao chép": gần ảnh huấn luyện hơn 99% ảnh kiểm tra
copy = (d_gan < thr).mean()
print("phân vị 1% · 10% · 50% của khoảng cách tới ảnh huấn luyện gần nhất:")
print("  mẫu của GAN: ", " · ".join(vn(v, 2) for v in np.quantile(d_gan, QS)))
print("  ảnh kiểm tra:", " · ".join(vn(v, 2) for v in np.quantile(d_test, QS)))
print(f"mẫu gần ảnh huấn luyện hơn 99% ảnh kiểm tra (dưới {vn(thr, 2)}): {pct(copy, 1)}")
ref = mn["runs"]["ns-r10-lr0.001"]["seeds"][0]["nn"]
assert np.allclose(np.quantile(d_test, QS), ref["test"], atol=2e-3), "khác với demo: hãy kiểm tra lại dữ liệu"
assert np.allclose(np.quantile(d_gan, QS), ref["sample"], atol=0.03) and copy <= 0.002

**Xa hơn.** Ảnh kiểm tra thật nằm gần ảnh huấn luyện hơn mẫu của GAN ở mọi phân vị, và không mẫu nào trong 1.000 mẫu
gần một ảnh huấn luyện hơn 99% ảnh kiểm tra. Với 55.000 ảnh, bộ sinh này không chép: mẫu của nó hơi lệch khỏi những
chữ số thật (nét méo, độ dày lạ), nên xa mọi ảnh huấn luyện hơn một chữ số thật mới.

Nhưng "không nhận ảnh thật trực tiếp" không ngăn được việc sao chép. $D$ học trên ảnh thật, và nếu tập huấn luyện nhỏ,
$D$ học thuộc nó rồi dạy lại cho $G$. Demo huấn luyện cùng cấu hình trên 100 và 1.000 ảnh đầu tiên. Ô dưới đọc tỉ lệ
sao chép của chúng, rồi tính lại khoảng cách tới ảnh huấn luyện gần nhất của 8 mẫu mà demo lưu từ GAN học trên 100 ảnh.

In [ ]:
for name in ("n100", "n1000", "ns-r10-lr0.001"):
    seeds = mn["runs"][name]["seeds"]
    n = mn["runs"][name]["cfg"]["ntrain"]
    print(f"học trên {vn(n, 0):>6} ảnh: {span([100 * s['copy'] for s in seeds], 1)}% mẫu sao chép (3 seed) · ngưỡng: "
          f"1% ảnh kiểm tra gần hơn {vn(seeds[0]['nn']['test'][0], 2)}")  # fmt: skip
d100, _ = nearest(Xte[:1000], Xtr[:100])  # ảnh kiểm tra → 100 ảnh huấn luyện
assert np.allclose(np.quantile(d100, QS), mn["runs"]["n100"]["seeds"][0]["nn"]["test"], atol=2e-3)
x8, near8 = u8(mn["nn"]["n100"]["x"]), u8(mn["nn"]["n100"]["near"])
d8, i8 = nearest(x8, Xtr[:100])
print("8 mẫu (GAN học trên 100 ảnh) → ảnh huấn luyện gần nhất:", " · ".join(vn(v, 2) for v in d8))
assert np.allclose(d8, mn["nn"]["n100"]["dist"], atol=0.01) and np.abs(Xtr[:100][i8] - near8).max() < 0.003
print(
    f"ngưỡng sao chép với 100 ảnh: {vn(np.quantile(d100, 0.01), 2)}; {(d8 < np.quantile(d100, 0.01)).sum()}/8 mẫu dưới ngưỡng"
)

fig, axes = plt.subplots(2, 8, figsize=(9, 2.8))
for k in range(8):
    axes[0, k].imshow(x8[k].reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
    axes[1, k].imshow(Xtr[:100][i8[k]].reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
    axes[1, k].set_title(f"cách {vn(d8[k], 1)}", fontsize=8)
    for a in axes[:, k]:
        a.set_xticks([])
        a.set_yticks([])
axes[0, 0].set_ylabel("mẫu", fontsize=9)
axes[1, 0].set_ylabel("ảnh học\ngần nhất", fontsize=9)
plt.tight_layout()
plt.show()

Hàng trên là 8 mẫu của GAN học trên 100 ảnh, hàng dưới là ảnh huấn luyện gần nhất của từng mẫu cùng khoảng cách. Phần
lớn các cặp gần như giống hệt nhau. Với 100 ảnh, khoảng một nửa số mẫu gần một ảnh huấn luyện hơn 99% ảnh kiểm tra;
với 1.000 ảnh còn gần 0, với 55.000 ảnh là 0. Carlini và cộng sự (2023) cũng trích được ảnh huấn luyện từ các GAN
lớn trên CIFAR-10, dù ít hơn từ mô hình khuếch tán cùng chất lượng.

## 7 · Lưới thí nghiệm trên MNIST: mất mát của G và R1

Demo huấn luyện một lưới cấu hình, 3 seed mỗi cấu hình, cùng hai mạng của mục 6. Mỗi lần chạy được đo trên 10.000 mẫu
bằng thước đo của Bài 19: **FD**, khoảng cách Fréchet giữa đặc trưng LeNet của mẫu và của 10.000 ảnh kiểm tra (cách
tính của FID, với LeNet của Bài 12 thay cho Inception, nên FD chỉ so được trong khóa học). Hai tập ảnh thật cho FD
1,32. Ô dưới in toàn bộ lưới; "hỏng" nghĩa là FD trên 100 (ảnh không còn là chữ số).

In [ ]:
def label(name):
    c = mn["runs"][name]["cfg"]
    parts = ["không bão hòa" if c["loss"] == "ns" else "mất mát gốc", f"R1 γ {c['r1']:g}" if c["r1"] else "không R1"]
    parts.append(f"η {c['lr']:g}".replace(".", ","))
    parts += [f"d = {c['d']}"] * (c["d"] != 16) + [f"{vn(c['ntrain'], 0)} ảnh"] * (c["ntrain"] < 55000)
    return ", ".join(parts + [f"β₁ {vn(c['b1'], 1)}"] * (c["b1"] != 0.5))


def fds(name):
    return [s["fd"] for s in mn["runs"][name]["seeds"] if not s.get("diverged")]


def failed(seeds):
    return sum(s.get("diverged", False) or s["fd"] > 100 for s in seeds)


print(f"{'cấu hình':>46} {'FD (3 seed)':>14} {'precision':>11} {'recall':>11} {'hỏng':>5}")
for name, r in mn["runs"].items():
    s = r["seeds"]
    print(f"{label(name):>46} {span(fds(name)):>14} {span([x['P'] for x in s]):>11} {span([x['R'] for x in s]):>11}"
          f" {failed(s)}/{len(s):>2}")  # fmt: skip
print(f"{'VAE d = 16 của Bài 19, z ~ N(0, I)':>46} {span([x['fd'] for x in mn['vae']['prior']['seeds']]):>14}")
print(f"{'VAE d = 16 của Bài 19, GMM khớp trên mã':>46} {span([x['fd'] for x in mn['vae']['gmm']['seeds']]):>14}")
print(f"{'hai tập ảnh thật':>46} {vn(mn['judge']['floor']):>14}")

Đếm riêng những lần chạy trên đúng cặp mạng của mục 6 (nhiễu 16 chiều, 55.000 ảnh, $\beta_1 = 0{,}5$), chỉ khác mất
mát của $G$, $\eta$ và R1:

In [ ]:
same = [
    n for n, r in mn["runs"].items() if r["cfg"]["d"] == 16 and r["cfg"]["ntrain"] == 55000 and r["cfg"]["b1"] == 0.5
]
for has_r1 in (False, True):
    names = [n for n in same if (mn["runs"][n]["cfg"]["r1"] > 0) == has_r1]
    seeds = [s for n in names for s in mn["runs"][n]["seeds"]]
    bad = [label(n) for n in names for s in mn["runs"][n]["seeds"] if failed([s])]
    print(f"{'có R1' if has_r1 else 'không R1'}: {failed(seeds)}/{len(seeds)} lần chạy hỏng · FD "
          f"{span([s['fd'] for s in seeds])}" + (f" · hỏng: {'; '.join(sorted(set(bad)))}" if bad else ""))  # fmt: skip
best = fds("ns-r10-lr0.001")
print(f"tốt nhất (không bão hòa, R1 γ 10, η 0,001): FD {span(best)}; VAE + GMM của Bài 19: "
      f"{span([x['fd'] for x in mn['vae']['gmm']['seeds']])}")  # fmt: skip

Không R1, 4/12 lần chạy hỏng, và cả 4 đều dùng mất mát gốc: ở $\eta = 0{,}001$ cả 3 seed kẹt ở FD 288–317, đúng
bộ sinh mà mục 3 thấy gradient nhỏ hơn 14.000 lần. Mất mát không bão hòa không hỏng lần nào, nhưng không R1 thì dừng
ở FD 23–33. Với R1, 0/18 lần chạy hỏng, cả với mất mát gốc; FD còn 7–11 với γ 1 và 10 (γ = 100 phạt quá mạnh: 16–17).
Phạt gradient làm phần lớn cái
"khó huấn luyện" biến mất. Bộ sinh tốt nhất có FD 7,88–8,02, ngang VAE + GMM của Bài 19 (7,88–8,94; trong dao động
giữa các seed), và tốt hơn hẳn VAE lấy $z$ từ $\mathcal N(0, I)$ (31,18–31,82), dù cùng một kiến trúc bộ giải mã.

Hai dòng khác đáng chú ý. $\beta_1 = 0{,}9$ (mặc định của Adam) không R1 hỏng cả 3 seed: Radford và cộng sự (2015)
viết rằng để $\beta_1$ ở 0,9 gây dao động và mất ổn định, và hạ xuống 0,5 giúp ổn định. Nhiễu chỉ 2 chiều ($d = 2$) cho
FD 20–31 và recall gần 0: hai số không đủ để tạo ra hết các kiểu chữ.

Hình dưới vẽ FD (trục log) theo bước huấn luyện của 4 cấu hình, mỗi đường một seed, cùng $D(G(z))$, xác suất mà $D$
lúc đó gán cho mẫu giả là thật.

In [ ]:
SHOW = [("ns-r10-lr0.001", BLUE, "-"), ("ns-r0-lr0.001", ORANGE, "--"), ("sat-r0-lr0.001", GREEN, ":"),
        ("sat-r10-lr0.001", GREY, "-.")]  # fmt: skip
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
for name, color, ls in SHOW:
    for k, s in enumerate(mn["runs"][name]["seeds"]):
        t_fd = [(t["step"], t["fd"]) for t in s["traj"] if "fd" in t]
        t_d = [(t["step"], t["DGz"]) for t in s["traj"] if t["step"] > 0]
        a1.plot(*zip(*t_fd), color=color, ls=ls, lw=1.6, label=label(name) if k == 0 else None)
        a2.plot(*zip(*t_d), color=color, ls=ls, lw=1.6)
a1.axhline(mn["judge"]["floor"], color="black", lw=0.8)
a1.text(20000, mn["judge"]["floor"] * 1.15, "hai tập ảnh thật", ha="right", fontsize=8)
a1.set(yscale="log", ylim=(1, 500), xlabel="bước", ylabel="FD (thấp là tốt)", title="FD của 5.000 mẫu, mỗi 1.000 bước")
a2.set(xscale="log", yscale="log", ylim=(1e-6, 1), xlabel="bước", ylabel="D(G(z))", title="D(ảnh giả) trong lúc học")
a2.axhline(0.5, color="black", lw=0.8)
fig.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, 0.02), ncol=2)
vn_axes(fig)
plt.show()
ld = [t["LD"] for s in mn["runs"]["ns-r10-lr0.001"]["seeds"] for t in s["traj"] if t["step"] >= 2000]
print(f"tốt nhất: mất mát của D từ bước 2.000 đến hết nằm trong {span(ld)} (đoán mò: 2·ln 2 = {vn(2 * math.log(2))})")
fd_at = {k: [next(t["fd"] for t in s["traj"] if t["step"] == k) for s in mn["runs"]["ns-r10-lr0.001"]["seeds"]]
         for k in (0, 2000)}  # fmt: skip
print(f"  trong khi FD đi từ {span(fd_at[0], 0)} (bước 0) qua {span(fd_at[2000], 1)} (bước 2.000) xuống {span(best)}"
      f" (bước 20.000): giảm thêm {vn(min(fd_at[2000]) / max(best), 1)}–{vn(max(fd_at[2000]) / min(best), 1)} lần")  # fmt: skip

Hai cấu hình có R1 (liền xanh, gạch chấm xám) giảm FD gần như suốt 20.000 bước, có dao động. Mất mát gốc không R1
(chấm xanh lá) đứng yên ở FD gần 300: bên phải, $D(G(z))$ của nó rơi xuống dưới $10^{-2}$ trong vài trăm đến 2.000
bước đầu và không bao giờ lên lại (tệp làm tròn $D(G(z))$ tới $10^{-6}$, nên đường đứt ở những mốc bằng 0). Mất mát
không bão hòa không R1 (gạch cam) thoát được, nhưng dừng ở FD trên 20.

Quan trọng hơn: ở lần chạy tốt nhất, mất mát của $D$ nằm sát $2 \ln 2 \approx 1{,}39$ (giá trị của một $D$ đoán mò
$\tfrac12$) từ bước 2.000 đến hết, trong khi FD còn giảm thêm khoảng 6 lần. **Mất mát của GAN không cho biết ảnh đã tốt
chưa**: phải đo từ bên ngoài.

## 8 · Đánh giá: $D \approx \tfrac12$ có nghĩa là thành công?

Ở cuối lần chạy tốt nhất, chính $D$ của GAN gán cho ảnh thật và ảnh giả gần như cùng một xác suất:

In [ ]:
for name in ("ns-r10-lr0.001", "sat-r0-lr0.001"):
    end = [s["traj"][-1] for s in mn["runs"][name]["seeds"]]
    print(f"{label(name):>32}, bước 20.000: D(ảnh huấn luyện) {span([t['Dtr'] for t in end], 3)} · "
          f"D(ảnh chưa thấy) {span([t['Dva'] for t in end], 3)} · D(G(z)) {span([t['DGz'] for t in end], 3)}")  # fmt: skip

Với bộ sinh tốt nhất, $D$ cho cả ảnh thật lẫn ảnh giả xác suất gần đúng ½. Nhìn qua, đó là điểm cân bằng của mục 4:
$D^* = \tfrac12$ khi $p_g = p_{\text{data}}$. Nhưng mục 4.1 đã cho thấy điều ngược lại không đúng: $D$ của *riêng
lần chạy này* bị qua mặt không có nghĩa là *không bộ phân biệt nào* phân biệt được.

Cách kiểm tra trực tiếp là **kiểm định hai mẫu bằng bộ phân loại** (C2ST; Lopez-Paz & Oquab, 2017): huấn luyện một mạng
mới, chưa từng tham gia trò chơi, để phân biệt 5.000 ảnh GAN với 5.000 ảnh thật, rồi đo độ chính xác trên 5.000 + 5.000
ảnh khác mà nó chưa thấy. 50% nghĩa là không phân biệt được. Như demo, bộ phân loại là MLP 784 → 128 → 1 (lớp `MLP` của
mục 4), Adam, batch 128, 10 lượt.

### 🤔 Dự đoán trước
Bộ sinh tốt nhất có FD 7,88–8,02 và $D \approx \tfrac12$. Bộ phân loại mới đúng khoảng bao nhiêu phần trăm: gần 50%,
khoảng 70%, hay gần 90%? Và với hai tập ảnh thật (10.000 ảnh huấn luyện so với 10.000 ảnh kiểm tra)?

In [ ]:
def c2st(A, Bset, seed=0, epochs=10, lr=1e-3, bs=128):
    """Độ chính xác trên nửa chưa thấy của một MLP 784 → 128 → 1 mới, học phân biệt A (nhãn 1) với Bset (nhãn 0)."""
    rng = np.random.default_rng(seed)
    n = min(len(A), len(Bset)) // 2
    A, Bset = A[rng.permutation(len(A))], Bset[rng.permutation(len(Bset))]  # xáo trước khi chia đôi
    Xa, Xb = np.concatenate([A[:n], Bset[:n]]), np.concatenate([A[n : 2 * n], Bset[n : 2 * n]])
    y = np.r_[np.ones(n), np.zeros(n)]
    net = MLP([784, 128, 1], 1, rng)
    for _ in range(epochs):
        order = rng.permutation(2 * n)
        for i in range(0, 2 * n, bs):
            idx = order[i : i + bs]
            hs = net.fwd(Xa[idx][None])
            d_out = (sigmoid(hs[-1][0, :, 0]) - y[idx]) / len(idx)  # cross-entropy nhị phân (Bài 04)
            net.adam(net.back(hs, d_out[None, :, None])[0], lr, b1=0.9)
    return float(((net.fwd(Xb[None])[-1][0, :, 0] > 0) == (y > 0.5)).mean())


real10k = train_img[np.random.default_rng(50).choice(55000, 10000, replace=False)].astype(np.float32) / 255
acc_gan, acc_real = c2st(Xg, Xte), c2st(real10k, Xte)
print(f"bộ phân loại mới, trên ảnh chưa thấy: GAN tốt nhất (seed 0) so với ảnh thật {pct(acc_gan, 1)} · "
      f"ảnh thật so với ảnh thật {pct(acc_real, 1)}")  # fmt: skip
assert 0.85 < acc_gan < 0.92 and abs(acc_real - 0.5323) < 0.01, "khác với demo: hãy kiểm tra lại dữ liệu"

Gần 90%. Bộ phân loại mới phân biệt ảnh GAN với ảnh thật dễ dàng, trong khi hai tập ảnh thật chỉ cho khoảng 53%. $D
\approx \tfrac12$ chỉ nghĩa là $D$ **hiện tại** đã bị qua mặt: $D$ và $G$ đuổi nhau, $D$ luôn chậm một nhịp, và R1 còn
cố ý giữ $D$ trơn. Một mạng mới, được học đến cùng trên một bài toán cố định, tìm ra những khác biệt mà $D$ đã bỏ qua.
(Bộ sinh ở đây là bản làm tròn mà demo chạy; demo đo trên trọng số gốc, nên con số hơi khác ở chữ số thứ ba.)

Bảng dưới đặt C2ST cạnh FD và precision/recall cho vài cấu hình, từ các lần chạy đầy đủ của demo (3 seed). **Precision**
(độ chính xác) là tỉ lệ mẫu nằm trong vùng của ảnh thật, **recall** (độ phủ) là tỉ lệ ảnh thật nằm trong vùng của mẫu,
cả hai trên đặc trưng LeNet với 3 láng giềng gần nhất (Kynkäänniemi và cộng sự, 2019). Precision cao mà recall thấp
là dấu hiệu đo được của mode collapse.

In [ ]:
rows = [("hai tập ảnh thật", mn["real"]), ("VAE d = 16, z ~ N(0, I) (Bài 19)", mn["vae"]["prior"]["seeds"]),
        ("VAE d = 16 + GMM (Bài 19)", mn["vae"]["gmm"]["seeds"])]  # fmt: skip
rows += [
    (label(n), mn["runs"][n]["seeds"])
    for n in ("ns-r10-lr0.001", "sat-r10-lr0.001", "ns-r0-lr0.001", "d2", "n1000", "n100")
]
print(f"{'':>44} {'FD':>12} {'C2ST':>12} {'precision':>11} {'recall':>11}")
for name, seeds in rows:
    print(f"{name:>44} {span([s['fd'] for s in seeds]):>12} {span([100 * s['c2st'] for s in seeds], 1) + '%':>12} "
          f"{span([s['P'] for s in seeds]):>11} {span([s['R'] for s in seeds]):>11}")  # fmt: skip

Ba điều đọc được từ bảng:
- **Không thước đo nào đủ một mình.** VAE + GMM của Bài 19 có FD ngang GAN tốt nhất, nhưng bộ phân loại mới chỉ đúng
  71–72% với nó so với 88–89% với GAN: FD nhìn đặc trưng của LeNet, C2ST nhìn thẳng vào điểm ảnh, và mỗi thước đo
  thấy một kiểu khác biệt. Precision của VAE + GMM cao hơn, recall của GAN cao hơn.
- **Recall thấp là mode collapse đo được.** Nhiễu 2 chiều cho precision bình thường nhưng recall gần 0: mẫu trông như
  chữ số, nhưng chỉ là vài kiểu chữ.
- **Sao chép đánh lừa FD.** GAN học trên 100 ảnh có precision cao nhất bảng (mẫu của nó *là* các ảnh thật, mục 6),
  nhưng recall 0,06–0,07 và C2ST 99%: 100 ảnh không đại diện cho cả phân phối.

### FD phụ thuộc vào số mẫu
FD so trung bình và hiệp phương sai của hai tập đặc trưng, ước lượng từ mẫu. Với ít mẫu, ước lượng hiệp phương sai
nhiễu hơn, và FD bị **lệch lên**: ngay cả hai tập ảnh thật cũng không cho 0 (Bińkowski và cộng sự, 2018). Ô dưới đọc
các phép đo của demo (5 lần lấy mẫu cho mỗi số mẫu), rồi lặp lại hiện tượng đó bằng một đặc trưng mà ta tự tính được:
32 thành phần chính (PCA, Bài 19) của 10.000 ảnh huấn luyện đầu tiên thay cho LeNet. Con số khác, hiện tượng như nhau.

In [ ]:
for n in mn["fdN"]["real"]:
    print(f"n = {vn(int(n), 0):>6}: FD hai tập ảnh thật {span(mn['fdN']['real'][n]):>9} · GAN tốt nhất "
          f"{span(mn['fdN']['gan'][n]):>9} (5 lần, đặc trưng LeNet, so với 10.000 ảnh kiểm tra)")  # fmt: skip


def fd(F1, F2):
    """Khoảng cách Fréchet giữa hai Gauss khớp trên hai tập đặc trưng: ‖m₁ − m₂‖² + tr(C₁ + C₂ − 2(C₁C₂)^½)."""
    C1, C2 = np.cov(F1, rowvar=False), np.cov(F2, rowvar=False)
    root = np.sqrt(np.clip(np.linalg.eigvals(C1 @ C2).real, 0, None)).sum()  # tr((C₁C₂)^½) qua trị riêng
    return float(((F1.mean(0) - F2.mean(0)) ** 2).sum() + np.trace(C1) + np.trace(C2) - 2 * root)


P10 = Xtr[:10000].astype(np.float64)
m_pca = P10.mean(0)
B_pca = np.linalg.eigh(np.cov(P10, rowvar=False))[1][:, ::-1][:, :32]  # 32 hướng có phương sai lớn nhất


def feat(X):
    return (X.astype(np.float64) - m_pca) @ B_pca


F_te, F_g = feat(Xte), feat(Xg)
NS = (1000, 2000, 5000, 10000)
fd_real = {n: [fd(feat(Xtr[np.random.default_rng(10 * n + k).choice(55000, n, replace=False)]), F_te) for k in range(3)]
           for n in NS}  # fmt: skip
fd_gan = {
    n: [fd(F_g[np.random.default_rng(n + k).choice(10000, n, replace=False)], F_te) for k in range(3)] for n in NS
}
for n in NS:
    print(f"PCA 32 chiều, n = {vn(n, 0):>6}: hai tập ảnh thật {span(fd_real[n], 3)} · GAN {span(fd_gan[n], 3)} (3 lần)")
mean_real = [np.mean(fd_real[n]) for n in NS]
lenet = [np.mean(mn["fdN"]["real"][str(n)]) for n in (1000, 10000)]
print(f"hai tập ảnh thật, 1.000 → 10.000 mẫu: FD giảm {vn(mean_real[0] / mean_real[-1], 1)} lần (PCA), "
      f"{vn(lenet[0] / lenet[1], 1)} lần (LeNet)")  # fmt: skip
assert all(a > b for a, b in itertools.pairwise(mean_real)) and mean_real[0] > 1.8 * mean_real[-1]

Với cả hai loại đặc trưng, FD của hai tập ảnh **thật** giảm 2,5–3,2 lần khi tăng từ 1.000 lên 10.000 mẫu: phần lệch
lên ấy là của ước lượng, không phải của dữ liệu. Vì vậy FD chỉ so được khi cùng số mẫu và cùng mạng đặc trưng. Mọi FD
của bài này dùng 10.000 mẫu và LeNet của Bài 12; FID trong các bài báo dùng Inception, thường với 50.000 mẫu (như
StyleGAN, Karras và cộng sự, 2019), nên không con số nào ở đây so được với FID công bố.

## 9 · Thử thay đổi

Các ô dưới chạy lại cùng các hàm với tham số khác. Bạn có thể sửa giá trị rồi chạy lại.

**Trò chơi một chiều:** chọn điểm xuất phát của riêng bạn. Thử bắt đầu ở xa ($\mu = -5$), hoặc với $s$ rất lớn.

In [ ]:
my_starts = [(-5.0, 0.5), (0.0, 0.3), (0.0, 3.0), (2.5, 2.5)]
p_sat, p_ns = flows(my_starts)
for (m0, s0), a_, b_ in zip(my_starts, p_sat[:, -1], p_ns[:, -1]):
    print(f"({vn(m0, 1)}; {vn(s0, 1)}) → mất mát gốc ({vn(a_[0], 2)}; {vn(a_[1], 2)}), JSD {vn(divergences(*a_)[0], 4)}"
          f" · không bão hòa ({vn(b_[0], 2)}; {vn(b_[1], 2)}), JSD {vn(divergences(*b_)[0], 4)}")  # fmt: skip

**Vòng 8 cụm với nhiễu 2 chiều:** demo cho thấy số chiều của nhiễu thay đổi hẳn bức tranh. Ô dưới chạy cấu hình không
bão hòa, không R1 của mục 5, chỉ đổi $z$ từ 16 xuống 2 chiều (3 lần chạy, 3.000 bước), rồi in kết quả lưu sẵn của demo
cho cùng cấu hình. Thử thêm `gammas=[1.0] * 3` hoặc `losses=["sat"] * 3`.

In [ ]:
_, _, log_dz2, _ = train_ring(["ns"] * 3, [0.0] * 3, 3000, dz=2, every=3000)
c_dz2 = log_dz2[-1]["counts"]
print(f"nhiễu 2 chiều, không R1, 3.000 bước: cụm có mẫu {(c_dz2 > 0).sum(1).tolist()} · đủ phần "
      f"{(c_dz2 >= 1000 / 32).sum(1).tolist()} · mẫu tốt {span(log_dz2[-1]['good'] * 100, 0)}%")  # fmt: skip
st = two["runs"]["ring-H32-ns-dz2-r0-alt-lr0.002"]["seeds"]
print(f"demo, 5 seed × 20.000 bước: cụm có mẫu {[r['final']['modes'] for r in st]} · đủ phần "
      f"{[r['final']['fair'] for r in st]} · mẫu tốt {span([100 * r['final']['hq'] for r in st], 0)}%")  # fmt: skip

Với nhiễu 2 chiều, bộ sinh trải mẫu ra nhiều cụm hơn hẳn so với nhiễu 16 chiều, kể cả không có R1. Đó là một kết quả
thực nghiệm trên dữ liệu này, không phải quy luật: trên MNIST thì ngược lại, nhiễu 2 chiều cho recall gần 0 (mục 8),
vì 2 số không đủ mô tả 10 chữ số với mọi kiểu viết. Một điều đúng với mọi số chiều: $G$ là hàm liên tục của $z$, nên
tập mẫu của nó liền một khối, và bộ sinh không thể đặt mẫu vào 8 cụm rời nhau mà không có mẫu nào nằm giữa chúng.

## 10 · Giới hạn và bước tiếp theo

- **Không có hàm mất mát để theo dõi.** Mất mát của $D$ đứng yên quanh $2\ln 2$ trong khi ảnh còn tốt lên nhiều lần;
  muốn biết mô hình tốt đến đâu phải đo từ bên ngoài, bằng nhiều thước đo.
- **Huấn luyện là một trò chơi, không phải một bài toán tối ưu.** Với mạng hữu hạn, cân bằng Nash có thể không tồn tại
  (Farnia & Ozdaglar, 2020); phạt gradient như R1 làm huấn luyện hội tụ cục bộ (Mescheder và cộng sự, 2018), nhưng
  không bảo đảm phủ hết dữ liệu.
- **Không có likelihood và không có bộ mã hóa.** GAN không cho $p(x)$ của một ảnh, và không có cách trực tiếp để tìm
  $z$ của một ảnh thật.

Mô hình khuếch tán (Bài 21) đã thay GAN ở phần lớn bài toán sinh ảnh: huấn luyện chỉ là một bài hồi quy ổn định, có
một mất mát giảm dần, và phủ dữ liệu tốt. Nhưng mất mát đối nghịch vẫn sống tiếp: autoencoder của Stable Diffusion
học với một bộ phân biệt trên từng mảng ảnh để ảnh nén không bị mờ (Rombach và cộng sự, 2022), và ADD (SDXL Turbo)
chưng cất mô hình khuếch tán bằng mất mát đối nghịch để sinh ảnh trong 1–4 bước (Sauer và cộng sự, 2023).

## Tóm tắt
- GAN có hai mạng: $D$ là bộ phân loại nhị phân (cross-entropy, Bài 04), $G$ học qua gradient đi xuyên qua $D$. Mất
  mát gốc $\log(1 - D(G(z)))$ bão hòa khi $D$ thắng dễ: trên MNIST, ở bước 500, gradient của nó nhỏ hơn khoảng
  14.000 lần so với mất mát không bão hòa $-\log D(G(z))$.
- Với $G$ cố định, $D^* = p_{\text{data}} / (p_{\text{data}} + p_g)$ và trò chơi thành $-\log 4 + 2\cdot\mathrm{JSD}$.
  Với $D^*$, mất mát gốc là gradient của JSD; mất mát không bão hòa là gradient của
  $\mathrm{KL}(p_g \,\|\, p_{\text{data}}) - 2\cdot\mathrm{JSD}$ (cả hai đã kiểm tra tới $10^{-6}$).
- Trên hai cụm Gauss, mất mát gốc phủ cả hai bằng một Gauss rộng $\mathcal N(0;\, 2{,}04^2)$ (cực tiểu JSD); mất mát
  không bão hòa chọn một cụm.
- Trên vòng 8 cụm, không R1 thì bộ sinh chỉ phủ vài cụm (mode collapse); có R1 thì phủ cả 8 và
  $D \approx \tfrac12$. Một "cụm đông nhất" đổi chỗ liên tục không nhất thiết là nhảy cụm: hãy kiểm tra còn mẫu tốt
  nào không.
- Trên MNIST: 4/12 lần chạy hỏng khi không có R1, 0/18 khi có. GAN tốt nhất có FD 7,88–8,02, ngang VAE + GMM của
  Bài 19.
- $D \approx \tfrac12$ không có nghĩa là thành công: một bộ phân loại mới phân biệt ảnh GAN với ảnh thật đúng 88–89%
  (hai tập ảnh thật: 53%). FD lệch theo số mẫu; recall thấp là mode collapse; với 100 ảnh huấn luyện, khoảng một nửa
  số mẫu là bản sao.

## Câu hỏi ôn tập
1. Vì sao mất mát gốc $\log(1 - D(G(z)))$ cho $G$ gần như không có gradient lúc đầu huấn luyện, còn $-\log D(G(z))$
   thì không?
2. Với $D^*$ tối ưu, $D^*(x) = \tfrac12$ ở những điểm nào? Vì sao "$D$ trung bình bằng ½" chưa đủ để kết luận
   $p_g = p_{\text{data}}$?
3. Trên hai cụm Gauss, vì sao mất mát không bão hòa thích một cụm hơn một Gauss rộng phủ cả hai, dù Gauss rộng có JSD
   nhỏ hơn?
4. R1 phạt gì, ở đâu? Vì sao nó giúp cả mất mát gốc, vốn hỏng 3/3 lần trên MNIST khi không có nó?
5. Bộ sinh không bao giờ nhận một ảnh thật. Vì sao nó vẫn chép ảnh huấn luyện khi chỉ có 100 ảnh?
6. Một GAN có FD 8, và một bộ phân loại mới phân biệt mẫu của nó với ảnh thật đúng 88%. Một VAE + GMM có FD 8 và
   C2ST 71–72%. Mô hình nào tốt hơn?

<details><summary>Gợi ý đáp án</summary>

1. Theo logit $a$ của mẫu giả, đạo hàm của $\log(1 - D)$ là $-D$: khi $D$ bác bỏ dễ dàng ($D \to 0$) nó tắt. Đạo hàm
   của $-\log D$ là $-(1 - D)$, gần 1 đúng lúc đó. Cùng hướng, cùng điểm cố định, khác độ lớn.
2. Chỉ nơi $p_g(x) = p_{\text{data}}(x)$: trong mục 4.1 là hai điểm. $D$ của một lần chạy có thể cho ½ trung bình vì
   nó chưa tối ưu (chậm một nhịp, bị R1 giữ trơn); một bộ phân loại mới vẫn phân biệt được 88%.
3. Với $D^*$, nó đi theo $\mathrm{KL}(p_g \,\|\, p_{\text{data}}) - 2\cdot\mathrm{JSD}$. KL theo chiều này phạt nặng
   việc đặt xác suất vào nơi không có dữ liệu (khoảng giữa hai cụm), và gần như không phạt việc bỏ sót một cụm.
4. Bình phương độ dốc $\lVert\nabla_x a(x)\rVert^2$ của $D$, chỉ trên ảnh thật. Nó giữ $D$ trơn, nên $D$ không thể
   bác bỏ mẫu giả với độ tự tin tuyệt đối ngay từ đầu; $D(G(z))$ không rơi về $10^{-5}$, và gradient của mất mát gốc
   không tắt.
5. $D$ học trên ảnh thật và có thể học thuộc 100 ảnh; gradient của nó kéo mẫu của $G$ về đúng những ảnh đó. Thông
   tin về dữ liệu vẫn đến được $G$, chỉ là gián tiếp.
6. Chưa nói được: hai thước đo nhìn hai thứ khác nhau (đặc trưng LeNet và điểm ảnh). Phải xem thêm precision, recall
   và ảnh huấn luyện gần nhất, rồi chọn theo việc mô hình sẽ được dùng.
</details>

## Tìm hiểu thêm
- T. Karras, S. Laine & T. Aila (2019), [StyleGAN](https://arxiv.org/abs/1812.04948): FID trên 50.000 ảnh.
- I. Goodfellow và cộng sự (2014), [Generative Adversarial Nets](https://arxiv.org/abs/1406.2661), và
  [NIPS 2016 Tutorial: Generative Adversarial Networks](https://arxiv.org/abs/1701.00160).
- M. Arjovsky & L. Bottou (2017), [Towards Principled Methods for Training GANs](https://arxiv.org/abs/1701.04862):
  Định lý 2.5 về mất mát không bão hòa.
- A. Radford, L. Metz & S. Chintala (2015), [DCGAN](https://arxiv.org/abs/1511.06434): Adam với $\beta_1 = 0{,}5$.
- L. Metz và cộng sự (2016), [Unrolled GANs](https://arxiv.org/abs/1611.02163): vòng 8 cụm và mode collapse.
- A. Srivastava và cộng sự (2017), [VEEGAN](https://arxiv.org/abs/1705.07761): đếm cụm và mẫu tốt trên dữ liệu 2 chiều.
- L. Mescheder, A. Geiger & S. Nowozin (2018),
  [Which Training Methods for GANs do actually Converge?](https://arxiv.org/abs/1801.04406): phạt gradient R1.
- F. Farnia & A. Ozdaglar (2020), [GANs May Have No Nash Equilibria](https://arxiv.org/abs/2002.09124).
- Y. Huang và cộng sự (2024), [The GAN is dead; long live the GAN! (R3GAN)](https://arxiv.org/abs/2501.05441):
  mất mát tương đối + R1 + R2 với kiến trúc hiện đại, vượt StyleGAN2.
- M. Heusel và cộng sự (2017), [FID](https://arxiv.org/abs/1706.08500); M. Bińkowski và cộng sự (2018),
  [Demystifying MMD GANs](https://arxiv.org/abs/1801.01401): FID lệch theo số mẫu, chỉ so được khi cùng số mẫu.
- T. Kynkäänniemi và cộng sự (2019), [Improved Precision and Recall](https://arxiv.org/abs/1904.06991).
- D. Lopez-Paz & M. Oquab (2017), [Revisiting Classifier Two-Sample Tests](https://arxiv.org/abs/1610.06545).
- N. Carlini và cộng sự (2023), [Extracting Training Data from Diffusion Models](https://arxiv.org/abs/2301.13188):
  mục 6 so sánh với GAN.
- M. Kang và cộng sự (2023), [GigaGAN](https://arxiv.org/abs/2303.05511): ảnh 512 px trong 0,13 giây.
- P. Esser, R. Rombach & B. Ommer (2021), [VQGAN](https://arxiv.org/abs/2012.09841); A. Sauer và cộng sự (2023),
  [Adversarial Diffusion Distillation](https://arxiv.org/abs/2311.17042) (SDXL Turbo).
- [GAN Lab](https://poloclub.github.io/ganlab/) (Kahng và cộng sự, 2018): GAN 2 chiều trong trình duyệt.

**Bài tiếp theo:** [Bài 21 · Khuếch tán & flow matching](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/diffusion/):
thay trò chơi bằng một bài hồi quy ổn định, học cách khử nhiễu từng bước.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")